# LLM Quantization and Quantization-Aware Training

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
# Install analysis/model-loading dependencies on Colab. Keep its compatible torch/CUDA build.
import os, sys, subprocess, pathlib, gc, json, time
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==5.18.0', 'datasets==4.8.5', 'timm==1.0.30',
                    'psutil', 'nvidia-ml-py', 'scikit-learn', 'tabulate',
                    'nbformat', 'nbconvert', 'ipykernel'], check=True)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
PROJECT = pathlib.Path('.').resolve()
pd.set_option('display.max_rows', 70)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:.5g}')
print('PyTorch', torch.__version__, 'CUDA available:', torch.cuda.is_available())


In [ ]:
"""Shared helpers: seeding, device selection, environment info, JSON persistence, serialized size."""


import io
import json
import os
import platform
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import torch

DEFAULT_SEED = 624

In [ ]:
def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def select_device(prefer: str | None = None) -> torch.device:
    """Pick cuda > mps > cpu unless a device string is forced."""
    if prefer:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [ ]:
def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

In [ ]:
def _run(cmd: list[str]) -> str | None:
    try:
        return subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=10).decode().strip()
    except Exception:
        return None

In [ ]:
def cpu_model_name() -> str:
    if sys.platform == "darwin":
        out = _run(["sysctl", "-n", "machdep.cpu.brand_string"])
        if out:
            return out
    if sys.platform.startswith("linux"):
        try:
            with open("/proc/cpuinfo") as f:
                for line in f:
                    if line.lower().startswith("model name"):
                        return line.split(":", 1)[1].strip()
        except Exception:
            pass
    return platform.processor() or platform.machine()

In [ ]:
def environment_info(device: torch.device) -> dict[str, Any]:
    import psutil

    info: dict[str, Any] = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "device": str(device),
        "cpu_model": cpu_model_name(),
        "cpu_physical_cores": psutil.cpu_count(logical=False),
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "system_ram_gb": round(psutil.virtual_memory().total / 1e9, 2),
        "torch_num_threads": torch.get_num_threads(),
    }
    try:
        import transformers

        info["transformers"] = transformers.__version__
    except Exception:
        pass
    try:
        import datasets

        info["datasets"] = datasets.__version__
    except Exception:
        pass
    try:
        import torchvision

        info["torchvision"] = torchvision.__version__
    except Exception:
        pass
    try:
        import timm

        info["timm"] = timm.__version__
    except Exception:
        pass
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        info["gpu_model"] = props.name
        info["gpu_total_memory_gb"] = round(props.total_memory / 1e9, 2)
        info["cuda"] = torch.version.cuda
        info["cudnn"] = torch.backends.cudnn.version()
    elif device.type == "mps":
        info["gpu_model"] = "Apple Silicon GPU (MPS, unified memory)"
        info["gpu_total_memory_gb"] = info["system_ram_gb"]
    else:
        info["gpu_model"] = "none (CPU inference)"
    return info

In [ ]:
def save_json(obj: Any, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)

In [ ]:
def load_json(path: str | Path) -> Any:
    with open(path) as f:
        return json.load(f)

In [ ]:
def _json_default(o: Any) -> Any:
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

In [ ]:
def state_dict_bytes(state: dict[str, torch.Tensor]) -> int:
    """Actual serialized size of a state dict (torch.save into memory). Includes tensor metadata."""
    buf = io.BytesIO()
    torch.save(state, buf)
    return buf.getbuffer().nbytes

In [ ]:
def unique_parameter_count(model: torch.nn.Module) -> int:
    """Count parameters once even when tied (e.g. GPT-2 lm_head and wte share storage)."""
    seen: set[int] = set()
    total = 0
    for p in model.parameters():
        key = p.data_ptr()
        if key in seen:
            continue
        seen.add(key)
        total += p.numel()
    return total

In [ ]:
class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.t0

In [ ]:
def results_dir(root: str | Path | None = None) -> Path:
    root = Path(root) if root else Path(os.environ.get("PA3_RESULTS", "results"))
    root.mkdir(parents=True, exist_ok=True)
    return root

In [ ]:
"""WikiText-2 block construction, calibration sampling and CIFAR-10 splits.

WikiText-2 protocol (assignment, Experimental Setup):
  * tokenize with the GPT-2 tokenizer
  * concatenate the nonempty examples of a split, separated by the end-of-text token
  * cut nonoverlapping blocks of SEQ_LEN tokens, discard the incomplete tail (count reported)
  * score the 127 next-token targets per block (the first token has no context)
"""


from dataclasses import dataclass

import numpy as np
import torch

SEQ_LEN = 128
N_CALIB = 128
CALIB_SEED = 624

In [ ]:
@dataclass
class BlockSplit:
    name: str
    blocks: torch.Tensor  # [n_blocks, SEQ_LEN] int64
    total_tokens: int
    discarded_tokens: int
    n_examples_used: int

    @property
    def n_blocks(self) -> int:
        return int(self.blocks.shape[0])

    @property
    def n_scored_targets(self) -> int:
        return self.n_blocks * (SEQ_LEN - 1)

    def summary(self) -> dict:
        return {
            "split": self.name,
            "nonempty_examples": self.n_examples_used,
            "total_tokens_incl_separators": self.total_tokens,
            "blocks": self.n_blocks,
            "seq_len": SEQ_LEN,
            "scored_targets": self.n_scored_targets,
            "discarded_tail_tokens": self.discarded_tokens,
        }

In [ ]:
def _tokenize_split(dataset_split, tokenizer, seq_len: int, name: str) -> BlockSplit:
    eot = tokenizer.eos_token_id
    ids: list[int] = []
    n_used = 0
    texts = [t for t in dataset_split["text"] if t.strip()]
    # Fast tokenizer batch call. Separator appended after every nonempty example.
    enc = tokenizer(texts, add_special_tokens=False)["input_ids"]
    for seq in enc:
        ids.extend(seq)
        ids.append(eot)
        n_used += 1
    total = len(ids)
    n_blocks = total // seq_len
    discarded = total - n_blocks * seq_len
    arr = torch.tensor(ids[: n_blocks * seq_len], dtype=torch.long).view(n_blocks, seq_len)
    return BlockSplit(name, arr, total, discarded, n_used)

In [ ]:
def load_wikitext_blocks(tokenizer, seq_len: int = SEQ_LEN, cache_dir: str | None = None) -> dict[str, BlockSplit]:
    from datasets import load_dataset

    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", cache_dir=cache_dir)
    return {split: _tokenize_split(ds[split], tokenizer, seq_len, split) for split in ("train", "validation", "test")}

In [ ]:
def sample_calibration(train: BlockSplit, n: int = N_CALIB, seed: int = CALIB_SEED) -> tuple[torch.Tensor, np.ndarray]:
    """Return n training blocks chosen with a fixed seed, plus the chosen indices (recorded for reproducibility)."""
    rng = np.random.RandomState(seed)
    idx = np.sort(rng.choice(train.n_blocks, size=n, replace=False))
    return train.blocks[idx].clone(), idx

In [ ]:
def iterate_batches(blocks: torch.Tensor, batch_size: int):
    for i in range(0, blocks.shape[0], batch_size):
        yield blocks[i : i + batch_size]

In [ ]:
# ----------------------------------------------------------------------------------------------
# CIFAR-10 (Task 3)
# ----------------------------------------------------------------------------------------------
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

In [ ]:
class _PILDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, transform):
        self.pairs, self.transform = pairs, transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img, label = self.pairs[i]
        return self.transform(img), label

In [ ]:
def _hf_cifar10(root: str):
    from datasets import load_dataset

    ds = load_dataset("uoft-cs/cifar10", cache_dir=root)
    tr = [(ex["img"], int(ex["label"])) for ex in ds["train"]]
    te = [(ex["img"], int(ex["label"])) for ex in ds["test"]]
    return tr, te

In [ ]:
def cifar10_loaders(root: str = "data", batch_size: int = 128, val_size: int = 5000, seed: int = 624,
                    num_workers: int = 2, source: str = "torchvision"):
    """Fixed train/validation split of the CIFAR-10 training set. The test set is held out for final evaluation.

    Preprocessing matches the edadaltocg/resnet18_cifar10 checkpoint (mean/std above, 32x32 input).
    Training augmentation: random crop with 4-pixel padding and horizontal flip. Evaluation: none.
    """
    import torchvision
    import torchvision.transforms as T
    from torch.utils.data import DataLoader, Subset

    norm = T.Normalize(CIFAR_MEAN, CIFAR_STD)
    train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), norm])
    eval_tf = T.Compose([T.ToTensor(), norm])

    if source == "torchvision":
        full_train_aug = torchvision.datasets.CIFAR10(root, train=True, download=True, transform=train_tf)
        full_train_eval = torchvision.datasets.CIFAR10(root, train=True, download=True, transform=eval_tf)
        test = torchvision.datasets.CIFAR10(root, train=False, download=True, transform=eval_tf)
    else:  # Hugging Face mirror (uoft-cs/cifar10), identical images and labels, faster on some networks
        tr, te = _hf_cifar10(root)
        full_train_aug, full_train_eval = _PILDataset(tr, train_tf), _PILDataset(tr, eval_tf)
        test = _PILDataset(te, eval_tf)

    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(full_train_aug))
    val_idx, train_idx = perm[:val_size], perm[val_size:]

    g = torch.Generator()
    g.manual_seed(seed)
    loaders = {
        "train": DataLoader(Subset(full_train_aug, train_idx), batch_size=batch_size, shuffle=True,
                            num_workers=num_workers, generator=g, drop_last=False),
        "train_eval": DataLoader(Subset(full_train_eval, train_idx), batch_size=256, shuffle=False, num_workers=num_workers),
        "val": DataLoader(Subset(full_train_eval, val_idx), batch_size=256, shuffle=False, num_workers=num_workers),
        "test": DataLoader(test, batch_size=256, shuffle=False, num_workers=num_workers),
    }
    split_info = {"train_size": len(train_idx), "val_size": len(val_idx), "test_size": len(test), "seed": seed,
                  "val_indices_head": val_idx[:10].tolist()}
    return loaders, split_info

In [ ]:
"""Uniform quantizers (Section 2.1 conventions) and the simulated-quantization projection module.

Conventions used everywhere in this package
-------------------------------------------
    q     = clip(round(r / S) + Z, q_min, q_max)
    r_hat = S * (q - Z)

Weights  : symmetric, per output channel, integer range [-Q_b, Q_b] with Q_b = 2^(b-1) - 1,
           S_W^(c) = max_i |W_ic| / Q_b, Z_W = 0.   (row-vector convention Y = X W + b, c = column of W)
Activations: asymmetric, per tensor, unsigned 8 bit, range includes zero,
           S_X = (x_max - x_min) / 255, Z_X = clip(round(-x_min / S_X), 0, 255).
Zero-range tensors: the scale is replaced by 1.0 so that q = Z and r_hat = 0 exactly.

Execution is *simulated*: tensors are quantized and immediately dequantized, and the matmul runs in FP32.
`integer_projection_reference` evaluates the derived integer form (int32/int64 accumulation, zero-point
correction, rescale, FP32 bias) and is used to verify that the simulated path is numerically identical.
"""


from dataclasses import dataclass

import torch
import torch.nn as nn

EPS = 1e-12


# ----------------------------------------------------------------------------------------------
# Weight quantization (symmetric, per output channel)
# ----------------------------------------------------------------------------------------------

In [ ]:
def weight_qrange(bits: int) -> int:
    return 2 ** (bits - 1) - 1

In [ ]:
def weight_scales(weight: torch.Tensor, bits: int, channel_dim: int = 1) -> torch.Tensor:
    """Per-output-channel symmetric scales. `channel_dim` is the output-channel axis.

    Conv1D (GPT-2): weight [in, out] -> channel_dim=1. nn.Linear / Conv2d: weight [out, ...] -> channel_dim=0.
    """
    qb = weight_qrange(bits)
    reduce_dims = [d for d in range(weight.dim()) if d != channel_dim]
    amax = weight.abs().amax(dim=reduce_dims) if reduce_dims else weight.abs()
    scale = amax / qb
    scale = torch.where(scale > 0, scale, torch.ones_like(scale))  # zero-range channel -> scale 1 (q = 0)
    return scale

In [ ]:
def _broadcast(scale: torch.Tensor, weight: torch.Tensor, channel_dim: int) -> torch.Tensor:
    shape = [1] * weight.dim()
    shape[channel_dim] = -1
    return scale.view(shape)

In [ ]:
def quantize_weight(weight: torch.Tensor, bits: int, channel_dim: int = 1, scale: torch.Tensor | None = None):
    """Return (q_int, scale, dequantized). `scale` may be supplied to reuse a fixed grid (GPTQ)."""
    qb = weight_qrange(bits)
    if scale is None:
        scale = weight_scales(weight, bits, channel_dim)
    s = _broadcast(scale, weight, channel_dim)
    q = torch.clamp(torch.round(weight / s), -qb, qb)
    deq = q * s
    return q.to(torch.int8), scale, deq

In [ ]:
def fake_quant_weight(weight: torch.Tensor, bits: int, channel_dim: int = 0) -> torch.Tensor:
    """Differentiable fake quantization with a straight-through estimator (used by QAT).

    Forward: clamp -> round -> dequantize. Backward: identity inside the clipping range (round is treated as
    identity), zero outside the range because `clamp` has zero gradient there.
    """
    qb = weight_qrange(bits)
    scale = weight_scales(weight.detach(), bits, channel_dim)
    s = _broadcast(scale, weight, channel_dim)
    lo, hi = -qb * s, qb * s
    inside = (weight >= lo) & (weight <= hi)
    w_c = torch.where(inside, weight, weight.detach().clamp(lo, hi))   # gradient 1 inside, 0 outside
    w_q = torch.round(w_c / s) * s
    return w_c + (w_q - w_c).detach()                                   # rounding treated as identity (STE)

In [ ]:
def activation_qparams(x_min: torch.Tensor, x_max: torch.Tensor, bits: int = 8):
    """Scale and zero point for an unsigned b-bit asymmetric quantizer. Range is widened to include 0."""
    qmax = 2**bits - 1
    x_min = torch.clamp(x_min, max=0.0)
    x_max = torch.clamp(x_max, min=0.0)
    scale = (x_max - x_min) / qmax
    scale = torch.where(scale > 0, scale, torch.ones_like(scale))
    zero = torch.clamp(torch.round(-x_min / scale), 0, qmax)
    return scale, zero

In [ ]:
def quantize_activation(x: torch.Tensor, scale: torch.Tensor, zero: torch.Tensor, bits: int = 8):
    qmax = 2**bits - 1
    q = torch.clamp(torch.round(x / scale) + zero, 0, qmax)
    return q, scale * (q - zero)

In [ ]:
def fake_quant_activation(x: torch.Tensor, x_min: torch.Tensor, x_max: torch.Tensor, bits: int = 8) -> torch.Tensor:
    """STE fake quantization for activations (QAT). Gradient 1 inside [x_min, x_max], 0 outside."""
    scale, zero = activation_qparams(x_min, x_max, bits)
    qmax = 2**bits - 1
    lo, hi = scale * (0 - zero), scale * (qmax - zero)
    inside = (x >= lo) & (x <= hi)
    x_c = torch.where(inside, x, x.detach().clamp(lo, hi))             # gradient 1 inside, 0 outside
    x_q = (torch.clamp(torch.round(x_c / scale) + zero, 0, qmax) - zero) * scale
    return x_c + (x_q - x_c).detach()                                   # rounding treated as identity (STE)

In [ ]:
class MinMaxObserver:
    """Aggregates the running min and max over every tensor it sees (static calibration, Section 2.1.1)."""

    def __init__(self):
        self.min: torch.Tensor | None = None
        self.max: torch.Tensor | None = None
        self.n_tensors = 0

    def update(self, x: torch.Tensor) -> None:
        mn, mx = x.detach().min(), x.detach().max()
        self.min = mn if self.min is None else torch.minimum(self.min, mn)
        self.max = mx if self.max is None else torch.maximum(self.max, mx)
        self.n_tensors += 1

    def qparams(self, bits: int = 8):
        assert self.min is not None, "observer never updated"
        return activation_qparams(self.min, self.max, bits)

In [ ]:
@dataclass
class QuantSpec:
    w_bits: int = 8
    a_bits: int = 8
    act_mode: str = "static"  # 'static' | 'dynamic' | 'none'

    @property
    def name(self) -> str:
        return f"{self.act_mode}_W{self.w_bits}A{self.a_bits}"

In [ ]:
class QuantProjection(nn.Module):
    """Simulated W{b}A8 projection Y = Q_A(X) Q_W(W) + b for a GPT-2 Conv1D (weight layout [in, out]).

    act_mode:
      'calibrate' : FP32 activations, observer records min/max         (static calibration pass)
      'static'    : fixed range from calibration                        (Static A8)
      'dynamic'   : range recomputed on the current input tensor       (Dynamic A8)
      'none'      : activations untouched (weight-only quantization)
    `runtime_in_scale` optionally divides the input by a per-channel vector (explicit SmoothQuant scaling
    for the MLP output projection, whose input comes from GELU and cannot be folded into a linear op).
    """

    def __init__(self, name: str, weight: torch.Tensor, bias: torch.Tensor, w_bits: int, a_bits: int = 8,
                 act_mode: str = "none", q_weight: torch.Tensor | None = None, scale: torch.Tensor | None = None,
                 runtime_in_scale: torch.Tensor | None = None):
        super().__init__()
        self.name = name
        self.in_features, self.out_features = weight.shape
        self.w_bits, self.a_bits = w_bits, a_bits
        self.act_mode = act_mode
        if q_weight is None:
            q_weight, scale, deq = quantize_weight(weight.detach(), w_bits, channel_dim=1, scale=scale)
        else:
            deq = q_weight.float() * scale.view(1, -1)
        self.register_buffer("q_weight", q_weight.to(torch.int8))
        self.register_buffer("w_scale", scale.detach().clone().float())
        self.register_buffer("weight", deq.detach().clone().float())  # dequantized FP32 copy used by the matmul
        self.bias = nn.Parameter(bias.detach().clone().float(), requires_grad=False)
        self.register_buffer("x_min", torch.tensor(0.0))
        self.register_buffer("x_max", torch.tensor(0.0))
        self.register_buffer("a_scale", torch.tensor(1.0))
        self.register_buffer("a_zero", torch.tensor(0.0))
        if runtime_in_scale is not None:
            self.register_buffer("runtime_in_scale_inv", (1.0 / runtime_in_scale).detach().clone().float())
        else:
            self.runtime_in_scale_inv = None
        self.observer = MinMaxObserver()
        self.track_clipping = False
        self.clip_stats = [0, 0]  # clipped elements, total elements

    # -- calibration ---------------------------------------------------------------------------
    def freeze_static_range(self) -> None:
        self.x_min = self.observer.min.detach().clone()
        self.x_max = self.observer.max.detach().clone()
        scale, zero = activation_qparams(self.x_min, self.x_max, self.a_bits)
        self.a_scale, self.a_zero = scale, zero

    # -- forward -------------------------------------------------------------------------------
    def quantize_input(self, x: torch.Tensor) -> torch.Tensor:
        if self.act_mode == "calibrate":
            self.observer.update(x)
            return x
        if self.act_mode == "none":
            return x
        if self.act_mode == "dynamic":
            scale, zero = activation_qparams(x.min(), x.max(), self.a_bits)
        else:  # static
            scale, zero = self.a_scale, self.a_zero
            if self.track_clipping:
                self.clip_stats[0] += int(((x < self.x_min) | (x > self.x_max)).sum())
                self.clip_stats[1] += x.numel()
        _, deq = quantize_activation(x, scale, zero, self.a_bits)
        return deq

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.runtime_in_scale_inv is not None:
            x = x * self.runtime_in_scale_inv
        x = self.quantize_input(x)
        size_out = x.size()[:-1] + (self.out_features,)
        y = torch.addmm(self.bias, x.reshape(-1, self.in_features), self.weight)
        return y.view(size_out)

    def extra_repr(self) -> str:
        return f"{self.name}: in={self.in_features}, out={self.out_features}, W{self.w_bits}A{self.a_bits} {self.act_mode}"

In [ ]:
def integer_projection_reference(x: torch.Tensor, proj: QuantProjection, scale: torch.Tensor | None = None,
                                 zero: torch.Tensor | None = None) -> torch.Tensor:
    """Evaluate Y = S_X * S_W^(c) * [ (q_X - Z_X) q_W ]_c + b with integer accumulation.

    (q_X - Z_X) is a signed integer in [-255, 255] and q_W in [-Q_b, Q_b]. The product accumulates exactly in
    int32 for d_in <= 3072 (|sum| <= 3072 * 255 * 127 < 2^31). We use int64 here for the CPU reference.
    The zero-point correction is folded in as sum_k (q_X - Z_X)_k q_W_kc = sum_k q_X_k q_W_kc - Z_X sum_k q_W_kc,
    where the second term is a per-column constant precomputed from the weights.
    """
    if scale is None:
        scale, zero = activation_qparams(x.min(), x.max(), proj.a_bits) if proj.act_mode == "dynamic" else (proj.a_scale, proj.a_zero)
    q_x, _ = quantize_activation(x.reshape(-1, proj.in_features), scale, zero, proj.a_bits)
    q_x = q_x.to(torch.int64)
    q_w = proj.q_weight.to(torch.int64)
    acc = q_x @ q_w                                   # int accumulation
    col_sum = q_w.sum(dim=0)                          # per output channel
    acc = acc - zero.to(torch.int64) * col_sum        # zero-point correction
    y = acc.to(torch.float32) * (scale * proj.w_scale.view(1, -1)) + proj.bias.view(1, -1)
    return y.view(*x.shape[:-1], proj.out_features)

In [ ]:
def pack_int4(q: torch.Tensor) -> torch.Tensor:
    """Pack signed 4-bit integers (range [-7, 7]) two per byte along the last dim (offset by +8 -> [1, 15])."""
    flat = (q.to(torch.int16) + 8).to(torch.uint8).reshape(-1)
    if flat.numel() % 2:
        flat = torch.cat([flat, torch.zeros(1, dtype=torch.uint8)])
    return (flat[0::2] | (flat[1::2] << 4)).contiguous()

In [ ]:
def unpack_int4(packed: torch.Tensor, numel: int, shape) -> torch.Tensor:
    lo = (packed & 0x0F).to(torch.int16) - 8
    hi = ((packed >> 4) & 0x0F).to(torch.int16) - 8
    flat = torch.stack([lo, hi], dim=1).reshape(-1)[:numel]
    return flat.to(torch.int8).reshape(shape)

In [ ]:
def export_quantized_state_dict(model: nn.Module) -> tuple[dict[str, torch.Tensor], dict]:
    """Integer export: int8 (or packed int4) weights + FP32 scales/zero-points/biases + all unquantized tensors.

    Returns the state dict and a metadata dict (layout, bits, packing, per-layer quantizer settings).
    """
    state: dict[str, torch.Tensor] = {}
    meta = {"layers": {}, "format": "symmetric per-output-channel weights, asymmetric per-tensor activations"}
    seen_ptr: set[int] = set()
    for name, module in model.named_modules():
        if isinstance(module, QuantProjection):
            q = module.q_weight.cpu()
            if module.w_bits <= 4:
                state[f"{name}.q_weight_packed"] = pack_int4(q)
                packed = True
            else:
                state[f"{name}.q_weight"] = q
                packed = False
            state[f"{name}.w_scale"] = module.w_scale.cpu()
            state[f"{name}.bias"] = module.bias.detach().cpu()
            if module.act_mode == "static":
                state[f"{name}.a_scale"] = module.a_scale.cpu()
                state[f"{name}.a_zero"] = module.a_zero.cpu().to(torch.uint8)
            if module.runtime_in_scale_inv is not None:
                state[f"{name}.runtime_in_scale_inv"] = module.runtime_in_scale_inv.cpu()
            meta["layers"][name] = {"w_bits": module.w_bits, "a_bits": module.a_bits, "act_mode": module.act_mode,
                                    "shape": list(q.shape), "packed": packed}
    # everything else (embeddings, LayerNorm, biases of unquantized modules) stays FP32, tied tensors once
    quant_prefixes = tuple(p + "." for p in meta["layers"].keys())
    for name, t in model.state_dict().items():
        if name.startswith(quant_prefixes):
            continue  # all QuantProjection tensors were exported explicitly above
        ptr = t.data_ptr()
        if ptr in seen_ptr and t.numel() > 1:
            continue  # tied tensor already stored
        seen_ptr.add(ptr)
        state[name] = t.detach().cpu()
    return state, meta

In [ ]:
def theoretical_packed_bytes(model: nn.Module) -> dict:
    """Bit-exact storage lower bound: quantized weights at w_bits per element, scales/zero-points FP32, rest FP32."""
    quant_bits = 0
    qparam_bytes = 0
    other_bytes = 0
    seen: set[int] = set()
    qnames = set()
    for name, m in model.named_modules():
        if isinstance(m, QuantProjection):
            qnames.add(name)
            quant_bits += m.q_weight.numel() * m.w_bits
            qparam_bytes += m.w_scale.numel() * 4 + (2 * 4 if m.act_mode == "static" else 0)
            other_bytes += m.bias.numel() * 4
            if m.runtime_in_scale_inv is not None:
                other_bytes += m.runtime_in_scale_inv.numel() * 4
    for name, t in model.state_dict().items():
        if any(name.startswith(q + ".") for q in qnames):
            continue
        if t.data_ptr() in seen and t.numel() > 1:
            continue
        seen.add(t.data_ptr())
        other_bytes += t.numel() * t.element_size()
    return {"quantized_weight_bytes": quant_bits / 8, "qparam_bytes": qparam_bytes, "fp32_other_bytes": other_bytes,
            "total_bytes": quant_bits / 8 + qparam_bytes + other_bytes}

In [ ]:
"""GPT-2 loading, projection access/replacement, language-model evaluation and analytic MAC counting."""


import copy
import math

import torch
import torch.nn as nn




MODEL_ID = "openai-community/gpt2"
PROJECTIONS = ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj")

In [ ]:
def load_gpt2(device: torch.device, model_id: str = MODEL_ID):
    from transformers import GPT2LMHeadModel, GPT2TokenizerFast

    tok = GPT2TokenizerFast.from_pretrained(model_id)
    model = GPT2LMHeadModel.from_pretrained(model_id, torch_dtype=torch.float32)
    model.config.use_cache = False
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad_(False)
    return model, tok

In [ ]:
def projection_names(n_layers: int = 12) -> list[str]:
    return [f"transformer.h.{i}.{p}" for i in range(n_layers) for p in PROJECTIONS]

In [ ]:
def get_module(model: nn.Module, name: str) -> nn.Module:
    m = model
    for part in name.split("."):
        m = getattr(m, part) if not part.isdigit() else m[int(part)]
    return m

In [ ]:
def set_module(model: nn.Module, name: str, new: nn.Module) -> None:
    parts = name.split(".")
    parent = get_module(model, ".".join(parts[:-1]))
    last = parts[-1]
    if last.isdigit():
        parent[int(last)] = new
    else:
        setattr(parent, last, new)

In [ ]:
def architecture_summary(model) -> dict:
    cfg = model.config
    blk = model.transformer.h[0]
    return {
        "n_layer": cfg.n_layer, "n_head": cfg.n_head, "n_embd": cfg.n_embd, "n_inner": 4 * cfg.n_embd,
        "vocab": cfg.vocab_size, "n_positions": cfg.n_positions,
        "lm_head_tied_to_wte": model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr(),
        "projection_types": {p: type(get_module(blk, p)).__name__ for p in PROJECTIONS},
        "projection_weight_shapes": {p: list(get_module(blk, p).weight.shape) for p in PROJECTIONS},
        "weight_layout": "Conv1D weight is [in_features, out_features]. Forward computes X @ W + b "
                         "(output channel = column). nn.Linear would be [out, in].",
    }

In [ ]:
def quantize_gpt2(base_model, spec: QuantSpec, q_weights: dict[str, tuple[torch.Tensor, torch.Tensor]] | None = None,
                  runtime_scales: dict[str, torch.Tensor] | None = None):
    """Deep-copy `base_model` and replace the 48 projections with QuantProjection modules.

    q_weights: optional {name: (q_int8, scale)} to install precomputed integer weights (GPTQ).
    runtime_scales: optional {name: s} explicit input scaling vectors (SmoothQuant on mlp.c_proj).
    """
    model = copy.deepcopy(base_model)
    for name in projection_names(model.config.n_layer):
        conv = get_module(model, name)
        # Conv1D stores weight as [in_features, out_features] and computes X @ W + b. Verify before choosing axes.
        assert conv.weight.shape[1] == conv.nf and conv.bias.shape[0] == conv.nf, f"unexpected Conv1D layout in {name}"
        q, s = (None, None) if q_weights is None else q_weights[name]
        rs = None if runtime_scales is None else runtime_scales.get(name)
        qp = QuantProjection(name, conv.weight.data, conv.bias.data, spec.w_bits, spec.a_bits,
                             act_mode=spec.act_mode, q_weight=q, scale=s, runtime_in_scale=rs)
        set_module(model, name, qp.to(conv.weight.device))
    return model

In [ ]:
def quant_modules(model) -> dict[str, QuantProjection]:
    return {n: m for n, m in model.named_modules() if isinstance(m, QuantProjection)}

In [ ]:
@torch.no_grad()
def calibrate_static_ranges(model, calib_blocks: torch.Tensor, batch_size: int, device: torch.device) -> None:
    """Static calibration: run the calibration set through the (weight-quantized) model with FP32 activations and
    aggregate min/max per projection input, then freeze the range (one scale + zero point per projection)."""
    mods = quant_modules(model)
    for m in mods.values():
        m.observer.__init__()
        m.act_mode = "calibrate"
    for batch in iterate_batches(calib_blocks, batch_size):
        model(input_ids=batch.to(device), use_cache=False)
    for m in mods.values():
        m.freeze_static_range()
        m.act_mode = "static"

In [ ]:
@torch.no_grad()
def evaluate_lm(model, blocks: torch.Tensor, batch_size: int, device: torch.device, max_blocks: int | None = None) -> dict:
    """Top-1 next-token accuracy, mean NLL and perplexity aggregated over all scored targets (127 per block)."""
    if max_blocks is not None:
        blocks = blocks[:max_blocks]
    total_nll = 0.0
    correct = 0
    n = 0
    for batch in iterate_batches(blocks, batch_size):
        batch = batch.to(device)
        logits = model(input_ids=batch, use_cache=False).logits[:, :-1, :].float()
        targets = batch[:, 1:]
        total_nll += float(torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), reduction="sum"))
        correct += int((logits.argmax(-1) == targets).sum())
        n += targets.numel()
    nll = total_nll / n
    return {"accuracy": 100.0 * correct / n, "nll": nll, "ppl": math.exp(nll), "scored_targets": n,
            "blocks": int(blocks.shape[0]), "batch_size": batch_size}

In [ ]:
def gpt2_mac_breakdown(model, seq_len: int = SEQ_LEN, batch_size: int = 1) -> dict:
    """MACs per forward pass of a full seq_len sequence (no cache). Convention: 1 MAC = 2 FLOPs.

    Counted: the four projections per block, the two attention matrix multiplications (QK^T and PV, over all
    heads), and the tied vocabulary head. Not counted as MACs (listed separately as 'other_elementwise_flops',
    estimated): LayerNorm, GELU, softmax, residual adds, bias adds, embedding lookups.
    """
    cfg = model.config
    L, d, h, V = seq_len, cfg.n_embd, cfg.n_head, cfg.vocab_size
    d_ff = 4 * d
    n = cfg.n_layer
    per_block = {
        "attn_projections": L * d * (3 * d) + L * d * d,      # c_attn + c_proj
        "attn_matmuls": 2 * L * L * d,                         # QK^T and PV (h heads x L x L x d/h each)
        "mlp_projections": L * d * d_ff + L * d_ff * d,        # c_fc + c_proj
    }
    total = {k: v * n * batch_size for k, v in per_block.items()}
    total["vocab_head"] = L * d * V * batch_size
    macs = sum(total.values())
    # elementwise estimate (FLOPs, not MACs): LN ~8/elem x (2n+1) x L d, GELU ~8/elem x n L d_ff, softmax ~5/elem x n h L L,
    # residual adds 2n L d, bias adds n L (3d + d + d_ff + d)
    other = batch_size * (8 * (2 * n + 1) * L * d + 8 * n * L * d_ff + 5 * n * h * L * L + 2 * n * L * d
                          + n * L * (3 * d + d + d_ff + d))
    return {"seq_len": L, "batch_size": batch_size, "macs_breakdown": total, "macs_total": macs,
            "flops_total": 2 * macs, "other_elementwise_flops_estimate": other,
            "convention": "1 MAC = 2 FLOPs (multiply + add). Matmul/projection MACs only. Elementwise ops listed separately"}

In [ ]:
"""Reconstruction errors, clipping statistics and weight / activation distribution analysis.

Normalized reconstruction error for a tensor A and its quantized-dequantized approximation A_hat:
    E_A = ||A - A_hat||_F^2 / (||A||_F^2 + eps)
Activation errors are accumulated over all rows of all analysis blocks (sum of squared errors over sum of
squared norms), so every method is scored on identical validation inputs taken from the FP32 model.
"""


from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

In [ ]:
def normalized_error(a: torch.Tensor, a_hat: torch.Tensor) -> float:
    a, a_hat = a.float(), a_hat.float()
    return float(((a - a_hat) ** 2).sum() / ((a**2).sum() + EPS))

In [ ]:
@torch.no_grad()
def layer_errors(fp_model, q_model, blocks: torch.Tensor, batch_size: int, device: torch.device,
                 names: list[str] | None = None) -> dict[str, dict]:
    """For every quantized projection compute, on the FP32 model's inputs X to that projection:
       E_W      weight error (original FP32 weight vs dequantized weight, in the quantizer's own domain),
       E_X      activation error of the tensor that is actually quantized (X, or X D^-1 for SmoothQuant),
       E_Y_w    output error from weight quantization alone    ||XW - X W_hat||^2 / ||XW||^2,
       E_Y      output error with weights and A8 activations   ||XW - Q(X) W_hat||^2 / ||XW||^2,
       clip     fraction of activation elements outside the static range (static A8 only).
    """
    names = names or projection_names(fp_model.config.n_layer)
    qmods = quant_modules(q_model)
    acc = {n: {"ex_num": 0.0, "ex_den": 0.0, "eyw_num": 0.0, "ey_num": 0.0, "ey_den": 0.0, "clip": 0, "count": 0}
           for n in names}
    handles = []

    def make_hook(name):
        qm: QuantProjection = qmods[name]

        def hook(module, inputs, output):
            x = inputs[0].reshape(-1, module.weight.shape[0]).float()
            y = output.reshape(-1, module.weight.shape[1]).float()
            s_fold = getattr(qm, "analysis_in_scale", None)
            x_in = x / s_fold.to(x.device) if s_fold is not None else x
            x_eff = x_in * qm.runtime_in_scale_inv if qm.runtime_in_scale_inv is not None else x_in
            if qm.act_mode == "dynamic":
                sc, z = activation_qparams(x_eff.min(), x_eff.max(), qm.a_bits)
                _, x_hat = quantize_activation(x_eff, sc, z, qm.a_bits)
            elif qm.act_mode == "static":
                _, x_hat = quantize_activation(x_eff, qm.a_scale, qm.a_zero, qm.a_bits)
                acc[name]["clip"] += int(((x_eff < qm.x_min) | (x_eff > qm.x_max)).sum())
            else:
                x_hat = x_eff
            a = acc[name]
            a["ex_num"] += float(((x_eff - x_hat) ** 2).sum())
            a["ex_den"] += float((x_eff**2).sum())
            a["count"] += x_eff.numel()
            y_w = torch.addmm(qm.bias, x_eff, qm.weight)          # weight quantization only
            y_q = torch.addmm(qm.bias, x_hat, qm.weight)          # weights + activations
            # Undo SmoothQuant's V-output reparameterization before comparing
            # with the ORIGINAL QKV output. Otherwise exact smoothing is counted
            # as quantization error, even with full precision weights/activations.
            output_scale = getattr(qm, "analysis_output_scale", None)
            if output_scale is not None:
                y_w = y_w * output_scale.to(y_w.device)
                y_q = y_q * output_scale.to(y_q.device)
            a["eyw_num"] += float(((y - y_w) ** 2).sum())
            a["ey_num"] += float(((y - y_q) ** 2).sum())
            a["ey_den"] += float((y**2).sum())

        return hook

    for n in names:
        handles.append(get_module(fp_model, n).register_forward_hook(make_hook(n)))
    try:
        for batch in iterate_batches(blocks, batch_size):
            fp_model(input_ids=batch.to(device), use_cache=False)
    finally:
        for h in handles:
            h.remove()

    out = {}
    for n in names:
        qm = qmods[n]
        w_ref = getattr(qm, "analysis_weight_ref", None)
        if w_ref is None:
            w_ref = get_module(fp_model, n).weight.data
        a = acc[n]
        out[n] = {
            "E_W": normalized_error(w_ref.to(qm.weight.device), qm.weight),
            "E_X": a["ex_num"] / (a["ex_den"] + EPS),
            "E_Y_w": a["eyw_num"] / (a["ey_den"] + EPS),
            "E_Y": a["ey_num"] / (a["ey_den"] + EPS),
            "clip_frac": a["clip"] / a["count"] if qm.act_mode == "static" else None,
        }
    return out

In [ ]:
def summarize_layer_errors(err: dict[str, dict]) -> dict:
    keys = ["E_W", "E_X", "E_Y_w", "E_Y"]
    summ = {k: float(np.mean([v[k] for v in err.values()])) for k in keys}
    clips = [v["clip_frac"] for v in err.values() if v["clip_frac"] is not None]
    summ["clip_frac"] = float(np.mean(clips)) if clips else None
    summ["per_type"] = {}
    for p in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
        vals = {k: float(np.mean([v[k] for n, v in err.items() if n.endswith(p)])) for k in keys}
        summ["per_type"][p] = vals
    return summ

In [ ]:
@torch.no_grad()
def collect_projection_stats(model, blocks: torch.Tensor, batch_size: int, device: torch.device,
                             layer_ids=(0, 5, 11), max_sample: int = 200_000, seed: int = 0,
                             input_transform: dict[str, torch.Tensor] | None = None) -> dict[str, dict]:
    """Per projection in the chosen blocks: per-input-channel max |X|, max |X|, 99th percentile of |X|, a value
    sample for histograms, and the same for the weight (per input channel max |W_j:|).
    `input_transform` optionally gives a per-channel divisor s so the statistics describe X' = X D^-1 (SmoothQuant
    analysis when the scaling is folded upstream in the analysed model).
    """
    names = [f"transformer.h.{i}.{p}" for i in layer_ids for p in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj")]
    gen = torch.Generator().manual_seed(seed)
    store = {n: {"ch_max": None, "abs_max": 0.0, "samples": [], "abs_samples": []} for n in names}
    handles = []

    def make_hook(name):
        def hook(module, inputs, output):
            x = inputs[0].reshape(-1, inputs[0].shape[-1]).float()
            if input_transform and name in input_transform:
                x = x / input_transform[name].to(x.device)
            cm = x.abs().amax(dim=0).cpu()
            st = store[name]
            st["ch_max"] = cm if st["ch_max"] is None else torch.maximum(st["ch_max"], cm)
            st["abs_max"] = max(st["abs_max"], float(cm.max()))
            k = min(max_sample // 8, x.numel())
            idx = torch.randint(0, x.numel(), (k,), generator=gen)
            st["samples"].append(x.reshape(-1)[idx.to(x.device)].cpu())
        return hook

    for n in names:
        handles.append(get_module(model, n).register_forward_hook(make_hook(n)))
    try:
        for batch in iterate_batches(blocks, batch_size):
            model(input_ids=batch.to(device), use_cache=False)
    finally:
        for h in handles:
            h.remove()

    out = {}
    for n in names:
        st = store[n]
        samples = torch.cat(st["samples"])[:max_sample]
        w = get_module(model, n).weight.data.float().cpu()  # [in, out] (Conv1D or QuantProjection dequantized)
        w_in_max = w.abs().amax(dim=1)
        ch = st["ch_max"]
        top = torch.topk(ch, 5)
        out[n] = {
            "act_abs_max": st["abs_max"],
            "act_p99_abs": float(torch.quantile(samples.abs(), 0.99)),
            "act_channel_max": ch.numpy(),
            "act_top5_channels": top.indices.tolist(),
            "act_top5_channel_max": top.values.tolist(),
            "act_channel_max_median": float(ch.median()),
            "act_dominance_ratio": float(ch.max() / (ch.median() + EPS)),
            "act_samples": samples.numpy(),
            "w_abs_max": float(w.abs().max()),
            "w_p99_abs": float(torch.quantile(w.abs().reshape(-1)[torch.randperm(w.numel(), generator=gen)[:max_sample]], 0.99)),
            "w_in_channel_max": w_in_max.numpy(),
            "w_dominance_ratio": float(w_in_max.max() / (w_in_max.median() + EPS)),
            "w_samples": w.reshape(-1)[torch.randperm(w.numel(), generator=gen)[:max_sample]].numpy(),
        }
    return out

In [ ]:
def stats_table(stats: dict[str, dict]) -> list[dict]:
    rows = []
    for n, s in stats.items():
        rows.append({"projection": n.replace("transformer.h.", "h"), "act_max": s["act_abs_max"], "act_p99": s["act_p99_abs"],
                     "act_max/median_ch": s["act_dominance_ratio"], "act_top_channels": s["act_top5_channels"][:3],
                     "w_max": s["w_abs_max"], "w_p99": s["w_p99_abs"], "w_max/median_ch": s["w_dominance_ratio"]})
    return rows

In [ ]:
def plot_distributions(stats_sets: dict[str, dict[str, dict]], outdir: str | Path, tag: str) -> list[str]:
    """One figure per projection: per-input-channel max |X| (log y) and histograms of X and W for each stats set
    (e.g. {'original': ..., 'smoothed a=0.5': ...}). Returns saved paths."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    paths = []
    first = next(iter(stats_sets.values()))
    for n in first:
        fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
        for label, st in stats_sets.items():
            s = st[n]
            axes[0].plot(s["act_channel_max"], lw=0.6, label=label)
            axes[1].plot(s["w_in_channel_max"], lw=0.6, label=label)
            axes[2].hist(s["act_samples"], bins=200, histtype="step", log=True, label=label)
            axes[3].hist(s["w_samples"], bins=200, histtype="step", log=True, label=label)
        axes[0].set_yscale("log")
        axes[0].set_title("per-input-channel max |X|")
        axes[0].set_xlabel("input channel")
        axes[1].set_yscale("log")
        axes[1].set_title("per-input-channel max |W_j:|")
        axes[1].set_xlabel("input channel")
        axes[2].set_title("activation values X")
        axes[3].set_title("weight values W")
        axes[0].legend(fontsize=7)
        fig.suptitle(f"{n}  ({tag})", fontsize=10)
        fig.tight_layout()
        p = outdir / f"dist_{tag}_{n.replace('transformer.h.', 'h').replace('.', '_')}.png"
        fig.savefig(p, dpi=110)
        plt.close(fig)
        paths.append(str(p))
    return paths

In [ ]:
@torch.no_grad()
def plot_quant_vs_original(q_model, fp_model, blocks: torch.Tensor, device: torch.device, name: str,
                           outdir: str | Path, tag: str, n_blocks: int = 4) -> str:
    """Histogram of the pre-quantization tensor and its quantized/dequantized approximation (weights and the
    activation actually quantized by `name`)."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    qm: QuantProjection = quant_modules(q_model)[name]
    w_ref = getattr(qm, "analysis_weight_ref", None)
    if w_ref is None:
        w_ref = get_module(fp_model, name).weight.data
    captured = {}

    def hook(module, inputs, output):
        captured["x"] = inputs[0].reshape(-1, inputs[0].shape[-1]).float()

    h = get_module(fp_model, name).register_forward_hook(hook)
    fp_model(input_ids=blocks[:n_blocks].to(device), use_cache=False)
    h.remove()
    x = captured["x"]
    s_fold = getattr(qm, "analysis_in_scale", None)
    x_eff = x / s_fold.to(x.device) if s_fold is not None else x
    if qm.runtime_in_scale_inv is not None:
        x_eff = x_eff * qm.runtime_in_scale_inv
    mode = qm.act_mode
    if mode == "dynamic":
        sc, z = activation_qparams(x_eff.min(), x_eff.max(), qm.a_bits)
    else:
        sc, z = qm.a_scale, qm.a_zero
    _, x_hat = quantize_activation(x_eff, sc, z, qm.a_bits)
    w_src = getattr(qm, "analysis_weight_transformed", None)
    w_src = w_src if w_src is not None else w_ref

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].hist(w_src.cpu().reshape(-1).numpy(), bins=200, histtype="step", log=True, label="W (pre-quant)")
    axes[0].hist(qm.weight.cpu().reshape(-1).numpy(), bins=200, histtype="step", log=True, label=f"W_hat (W{qm.w_bits})")
    axes[0].set_title("weights")
    axes[0].legend(fontsize=7)
    axes[1].hist(x_eff.cpu().reshape(-1).numpy(), bins=200, histtype="step", log=True, label="X (pre-quant)")
    axes[1].hist(x_hat.cpu().reshape(-1).numpy(), bins=200, histtype="step", log=True, label=f"X_hat (A{qm.a_bits} {mode})")
    axes[1].set_title("activations")
    axes[1].legend(fontsize=7)
    fig.suptitle(f"{name} ({tag})", fontsize=10)
    fig.tight_layout()
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    p = outdir / f"qvo_{tag}_{name.replace('transformer.h.', 'h').replace('.', '_')}.png"
    fig.savefig(p, dpi=110)
    plt.close(fig)
    return str(p)

In [ ]:
"""SmoothQuant (Xiao et al., 2023) implemented from the equations in Section 2.2.

For a projection Y = X W + b with X in R^{T x d_in}, choose a positive diagonal D = diag(s) and set
    X' = X D^-1,   W' = D W,   so that  X' W' + b = X W + b   (exact in real arithmetic).
    s_j = (max_t |X_tj| + eps)^alpha / (max_c |W_jc| + eps)^(1 - alpha)
Activation maxima come from the calibration set only.

Where the scales live in GPT-2 (per block i):
  attn.c_attn : input is ln_1(h).  D^-1 is folded into ln_1 (gamma/s, beta/s). W' = D W in c_attn.
  mlp.c_fc    : input is ln_2(h).  D^-1 folded into ln_2. W' = D W in c_fc.
  attn.c_proj : input is the merged attention context  softmax(QK^T) V.  Scaling V's output channels by 1/s
                scales the context by 1/s, so D^-1 is folded into the V slice of c_attn (columns 2d:3d of its
                weight and bias). W' = D W in attn.c_proj.
  mlp.c_proj  : input is GELU(c_fc(x)), a nonlinearity, so nothing upstream is linear. The scaling is applied
                explicitly at runtime (x * (1/s)) inside the projection module and is included in profiling.
All branches that share an input are consistent: ln_1 feeds only c_attn, ln_2 feeds only c_fc, the V slice is
used only by the context, and the residual stream is never scaled.
"""


import copy

import torch
import torch.nn as nn





SMOOTH_EPS = 1e-5

In [ ]:
class ScaledInputConv1D(nn.Module):
    """Conv1D-compatible projection that divides its input by a per-channel vector s before X @ W + b.
    Used for mlp.c_proj (explicit runtime smoothing)."""

    def __init__(self, conv, s: torch.Tensor):
        super().__init__()
        self.weight = nn.Parameter(conv.weight.data.clone(), requires_grad=False)
        self.bias = nn.Parameter(conv.bias.data.clone(), requires_grad=False)
        self.register_buffer("in_scale", s.clone())
        self.register_buffer("in_scale_inv", (1.0 / s).clone())
        self.nf = conv.nf

    def forward(self, x):
        x = x * self.in_scale_inv
        size_out = x.size()[:-1] + (self.nf,)
        return torch.addmm(self.bias, x.view(-1, x.size(-1)), self.weight).view(size_out)

In [ ]:
@torch.no_grad()
def collect_channel_maxima(model, calib_blocks: torch.Tensor, batch_size: int, device: torch.device) -> dict[str, torch.Tensor]:
    """Per-input-channel max |X| of every projection input over the calibration set (FP32 model)."""
    names = projection_names(model.config.n_layer)
    maxima: dict[str, torch.Tensor] = {}
    handles = []

    def make_hook(name):
        def hook(module, inputs, output):
            cm = inputs[0].reshape(-1, inputs[0].shape[-1]).abs().amax(dim=0).float()
            maxima[name] = cm if name not in maxima else torch.maximum(maxima[name], cm)
        return hook

    for n in names:
        handles.append(get_module(model, n).register_forward_hook(make_hook(n)))
    try:
        for batch in iterate_batches(calib_blocks, batch_size):
            model(input_ids=batch.to(device), use_cache=False)
    finally:
        for h in handles:
            h.remove()
    return maxima

In [ ]:
def compute_smooth_scales(model, act_max: dict[str, torch.Tensor], alpha: float, eps: float = SMOOTH_EPS) -> dict[str, torch.Tensor]:
    scales = {}
    for name, xm in act_max.items():
        w = get_module(model, name).weight.data  # [in, out]
        wm = w.abs().amax(dim=1).float()         # max over output channels for each input channel j
        s = (xm.to(w.device) + eps).pow(alpha) / (wm + eps).pow(1 - alpha)
        scales[name] = s.clamp(min=1e-5)
    return scales

In [ ]:
@torch.no_grad()
def apply_smoothing(fp_model, scales: dict[str, torch.Tensor]):
    """Return a deep copy of the FP32 model with the equivalent transformation folded in as described above."""
    model = copy.deepcopy(fp_model)
    d = model.config.n_embd
    for i, block in enumerate(model.transformer.h):
        pre = f"transformer.h.{i}."
        # c_attn <- ln_1
        s = scales[pre + "attn.c_attn"]
        block.ln_1.weight.div_(s)
        block.ln_1.bias.div_(s)
        block.attn.c_attn.weight.mul_(s.view(-1, 1))
        # c_fc <- ln_2
        s = scales[pre + "mlp.c_fc"]
        block.ln_2.weight.div_(s)
        block.ln_2.bias.div_(s)
        block.mlp.c_fc.weight.mul_(s.view(-1, 1))
        # attn.c_proj <- V slice of c_attn (columns 2d:3d)
        s = scales[pre + "attn.c_proj"]
        block.attn.c_attn.weight[:, 2 * d : 3 * d].div_(s.view(1, -1))
        block.attn.c_attn.bias[2 * d : 3 * d].div_(s)
        block.attn.c_proj.weight.mul_(s.view(-1, 1))
        # mlp.c_proj <- explicit runtime scaling
        s = scales[pre + "mlp.c_proj"]
        conv = block.mlp.c_proj
        conv.weight.mul_(s.view(-1, 1))
        block.mlp.c_proj = ScaledInputConv1D(conv, s).to(conv.weight.device)
    return model

In [ ]:
@torch.no_grad()
def verify_equivalence(fp_model, smoothed_model, blocks: torch.Tensor, device: torch.device, n_blocks: int = 8) -> dict:
    x = blocks[:n_blocks].to(device)
    y0 = fp_model(input_ids=x, use_cache=False).logits.float()
    y1 = smoothed_model(input_ids=x, use_cache=False).logits.float()
    diff = (y0 - y1).abs()
    return {"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean()),
            "relative_output_error": float(diff.norm() / (y0.norm() + 1e-12)), "n_blocks": n_blocks,
            "logits_abs_max": float(y0.abs().max())}

In [ ]:
def quantize_smoothed(smoothed_model, fp_model, scales: dict[str, torch.Tensor], spec: QuantSpec):
    """Quantize the smoothed model with the Section 2.1 quantizers and attach analysis metadata so that
    reconstruction errors are measured in the transformed domain (X' = X D^-1, W' = D W)."""
    runtime = {}
    for name in projection_names(smoothed_model.config.n_layer):
        m = get_module(smoothed_model, name)
        if isinstance(m, ScaledInputConv1D):
            runtime[name] = m.in_scale
    q_model = quantize_gpt2(smoothed_model, spec, runtime_scales=runtime)
    for name, qm in quant_modules(q_model).items():
        if name not in runtime:
            qm.analysis_in_scale = scales[name].detach().cpu().clone()
        qm.analysis_weight_ref = get_module(smoothed_model, name).weight.data.detach().cpu().clone()
        qm.analysis_weight_transformed = qm.analysis_weight_ref
        if name.endswith("attn.c_attn"):
            output_name = name.replace("attn.c_attn", "attn.c_proj")
            v_scale = scales[output_name].detach().cpu()
            qm.analysis_output_scale = torch.cat([torch.ones(2 * v_scale.numel()), v_scale])
    return q_model

In [ ]:
"""GPTQ (Frantar et al., 2023) implemented from the paper (Algorithm 1) for GPT-2 Conv1D projections.

Objective per layer, with calibration inputs X (rows = tokens) and weight W (Conv1D layout [d_in, d_out]):
    min_{W_hat} || X W - X W_hat ||_F^2 ,    H = 2 X^T X + lambda I   (d_in x d_in, shared by all output channels)
Each output channel c (a column of W, i.e. a row of W^T) is an independent least-squares problem with the same
Hessian, so all columns are processed together as the matrix W^T in R^{d_out x d_in}.

Algorithm (OBQ-style, fixed natural order, lazy block updates):
  1. H = 2/N sum_n x_n x_n^T over all calibration tokens. Dead inputs (H_ii = 0) get H_ii = 1 and W_i: = 0.
  2. Damping: H += lambda I with lambda = percdamp * mean(diag(H))   (percdamp = 0.01).
  3. Hinv = chol(H)^-T chol(H)^-1, then take U = chol(Hinv) upper whose diagonal reflects successive inverse-Hessian elimination. The
     error propagation for column i uses row i of U restricted to the not-yet-quantized columns.
  4. For column blocks of size B (default 128): for each column i inside the block
        q_i   = quant(w_i)                                   (per-output-channel scale fixed from the ORIGINAL W)
        err_i = (w_i - q_i) / U_ii
        w_j  -= err_i * U_ij   for j > i inside the block     (compensation of remaining weights)
     after the block: W[:, i2:] -= Err_block @ U[i1:i2, i2:]  (lazy batch update of the remaining columns)
Why this compensates: the OBS update that keeps the layer output optimal after fixing weight i to q_i is
     delta_w = -(w_i - q_i) / [H^-1]_ii * H^-1[:, i],
and the Cholesky form applies exactly this update to the still-free coordinates in a numerically stable way
(each step is a rank-1 update that corresponds to removing row/column i from H^-1).
Layers are processed sequentially. The calibration inputs of each layer come from the model in which all preceding
layers have already been quantized (so later layers compensate for earlier errors).
"""


import math
import time

import torch
import torch.nn as nn

In [ ]:
class GPTQLayer:
    """Accumulates H for one projection and runs the quantization/compensation loop."""

    def __init__(self, name: str, weight: torch.Tensor, w_bits: int, block_size: int = 128, percdamp: float = 0.01,
                 act_order: bool = False, work_device: torch.device | None = None):
        self.name = name
        self.w_bits = w_bits
        self.block_size = block_size
        self.percdamp = percdamp
        self.act_order = act_order
        self.dev = work_device or torch.device("cpu")
        self.W = weight.detach().t().contiguous().float().to(self.dev)   # [d_out, d_in]
        self.d_out, self.d_in = self.W.shape
        self.H = torch.zeros(self.d_in, self.d_in, dtype=torch.float32, device=self.dev)
        self.n = 0
        # Same integer grid as linear PTQ: per-output-channel scales from the original weights.
        self.scale = weight_scales(weight.detach().float(), w_bits, channel_dim=1).to(self.dev)  # [d_out]

    @torch.no_grad()
    def add_batch(self, x: torch.Tensor) -> None:
        x = x.reshape(-1, x.shape[-1]).float().to(self.dev)
        m = x.shape[0]
        self.H *= self.n / (self.n + m)
        self.n += m
        x = math.sqrt(2.0 / self.n) * x
        self.H += x.t() @ x

    @torch.no_grad()
    def quantize(self) -> dict:
        t0 = time.perf_counter()
        W = self.W.clone()
        H = self.H.clone()
        qb = 2 ** (self.w_bits - 1) - 1
        dead = torch.diag(H) == 0
        H[dead, dead] = 1.0
        W[:, dead] = 0.0
        perm = None
        if self.act_order:
            perm = torch.argsort(torch.diag(H), descending=True)
            W, H = W[:, perm], H[perm][:, perm]
        damp = self.percdamp * torch.mean(torch.diag(H))
        H += torch.eye(self.d_in, device=self.dev) * damp
        # Inverse via Cholesky. If H is still not PD, increase damping (numerical stabilization).
        tries = 0
        while True:
            try:
                L = torch.linalg.cholesky(H)
                Hinv = torch.cholesky_inverse(L)
                U = torch.linalg.cholesky(Hinv, upper=True)
                break
            except Exception:
                tries += 1
                if tries > 5:
                    raise
                H += torch.eye(self.d_in, device=self.dev) * damp * (10**tries)
        Q = torch.zeros_like(W)
        losses = torch.zeros(self.d_out, device=self.dev)
        s = self.scale.view(-1, 1)
        for i1 in range(0, self.d_in, self.block_size):
            i2 = min(i1 + self.block_size, self.d_in)
            cnt = i2 - i1
            W1 = W[:, i1:i2].clone()
            Q1 = torch.zeros_like(W1)
            Err1 = torch.zeros_like(W1)
            U1 = U[i1:i2, i1:i2]
            for i in range(cnt):
                w = W1[:, i]
                dinv = U1[i, i]
                q = torch.clamp(torch.round(w.view(-1, 1) / s), -qb, qb) * s
                q = q.view(-1)
                Q1[:, i] = q
                losses += (w - q) ** 2 / dinv**2 / 2
                err = (w - q) / dinv
                W1[:, i:] -= err.view(-1, 1) @ U1[i, i:].view(1, -1)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ U[i1:i2, i2:]
        if perm is not None:
            inv = torch.argsort(perm)
            Q = Q[:, inv]
        q_int = torch.clamp(torch.round(Q / s), -qb, qb).to(torch.int8)  # exact: Q is already on the grid
        self.q_weight = q_int.t().contiguous()        # back to Conv1D layout [d_in, d_out]
        self.deq_weight = (q_int.float() * s).t().contiguous()
        info = {"name": self.name, "seconds": time.perf_counter() - t0, "damp": float(damp),
                "block_size": self.block_size, "act_order": self.act_order, "cholesky_retries": tries,
                "dead_inputs": int(dead.sum()), "objective_proxy": float(losses.sum()), "tokens": self.n}
        return info

In [ ]:
class _Catcher(nn.Module):
    """Stops the forward pass at block 0 and records its (hidden_states, args, kwargs)."""

    def __init__(self, block):
        super().__init__()
        self.block = block
        self.inputs: list[tuple] = []

    def forward(self, hidden_states, *args, **kwargs):
        self.inputs.append((hidden_states.detach(), args, kwargs))
        raise _Stop

In [ ]:
class _Stop(Exception):
    pass

In [ ]:
@torch.no_grad()
def gptq_quantize_gpt2(fp_model, calib_blocks: torch.Tensor, batch_size: int, device: torch.device, w_bits: int,
                       block_size: int = 128, percdamp: float = 0.01, act_order: bool = False,
                       work_device: torch.device | None = None, verbose: bool = True):
    """Sequential GPTQ over the 12 blocks. Returns ({name: (q_int8, scale)}, info dict).

    Within a block the projections are quantized in execution order (c_attn, attn.c_proj, c_fc, mlp.c_proj) and each
    one collects its calibration inputs with the previously quantized projections of the same block already in
    place (true sequential processing). Weights of a block are replaced in a working copy of the model so the next
    block sees quantized upstream activations. Activations stay FP32 during GPTQ (A8 is applied afterwards).
    """
    import copy

    t_all = time.perf_counter()

    sampler = ResourceSampler(device, interval=0.02)
    sampler.start()
    model = copy.deepcopy(fp_model).to(device).eval()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    n_layer = model.config.n_layer
    blocks = model.transformer.h
    work_device = work_device or (device if device.type == "cuda" else torch.device("cpu"))

    # 1. capture inputs to block 0
    catcher = _Catcher(blocks[0])
    blocks[0] = catcher
    for b in range(0, calib_blocks.shape[0], batch_size):
        try:
            model(input_ids=calib_blocks[b : b + batch_size].to(device), use_cache=False)
        except _Stop:
            pass
    blocks[0] = catcher.block
    inputs = catcher.inputs  # list of (hidden [B, L, d], args, kwargs)

    results: dict[str, tuple[torch.Tensor, torch.Tensor]] = {}
    layer_info = []
    for i in range(n_layer):
        block = blocks[i]
        for p in PROJECTIONS:
            name = f"transformer.h.{i}.{p}"
            conv = get_module(block, p)
            g = GPTQLayer(name, conv.weight.data, w_bits, block_size, percdamp, act_order, work_device)
            h = conv.register_forward_hook(lambda m, inp, out, g=g: g.add_batch(inp[0]))
            for hs, args, kwargs in inputs:
                block(hs, *args, **kwargs)
            h.remove()
            info = g.quantize()
            conv.weight.data.copy_(g.deq_weight.to(conv.weight.device))   # quantized weights in place
            results[name] = (g.q_weight.cpu(), g.scale.cpu())
            layer_info.append(info)
            if verbose:
                print(f"  GPTQ {name}: {info['seconds']:.1f}s  damp={info['damp']:.3e}")
        # 2. propagate calibration activations through the quantized block
        next_inputs = []
        for hs, args, kwargs in inputs:
            output = block(hs, *args, **kwargs)
            hidden = output[0] if isinstance(output, (tuple, list)) else output
            next_inputs.append((hidden.detach(), args, kwargs))
        inputs = next_inputs
    summary = {"w_bits": w_bits, "block_size": block_size, "percdamp": percdamp, "act_order": act_order,
               "ordering": "activation-order (descending diag H)" if act_order else "natural column order",
               "total_seconds": time.perf_counter() - t_all, "n_calib_sequences": int(calib_blocks.shape[0]),
               "work_device": str(work_device), "layers": layer_info}
    if device.type == "cuda":
        summary["peak_gpu_allocated_mb"] = torch.cuda.max_memory_allocated(device) / 2**20
    sampler.stop()
    summary["peak_process_rss_mb"] = max(s["rss"] for s in sampler.samples) / 2**20
    summary["cpu_peak_method"] = "process RSS sampled during complete GPTQ stage every 0.02 s"
    del model
    return results, summary

In [ ]:
def build_gptq_model(fp_model, q_weights: dict[str, tuple[torch.Tensor, torch.Tensor]], spec: QuantSpec):
    """Install GPTQ integer weights into QuantProjection modules with the chosen activation quantizer."""
    dev = next(fp_model.parameters()).device
    qw = {n: (q.to(dev), s.to(dev)) for n, (q, s) in q_weights.items()}
    return quantize_gpt2(fp_model, spec, q_weights=qw)

In [ ]:
"""Profiling methodology shared by all GPT-2 and ResNet-18 configurations (Section 1.3).

  * latency: `warmup` untimed passes then `iters` timed passes, batch size fixed by the caller (1), device
    synchronized before and after every pass. Wall clock via time.perf_counter. No profiler attached.
  * throughput: sequences (or images) per second and input tokens per second from the mean latency.
  * profiler: torch.profiler on a few extra passes, used only for operator-level breakdown and its FLOP estimate.
  * memory: peak GPU allocated bytes (torch.cuda.max_memory_allocated. MPS driver/current allocated). Peak
    process RSS sampled by a background thread (psutil) plus ru_maxrss.
  * utilization: psutil process and system CPU percent sampled every `sample_interval` s. NVIDIA GPU
    utilization and board power via NVML (device wide, not per process) when available.
  * power/energy: GPU power integrated over the sampled interval of a repeated-pass loop that is extended to at
    least `min_power_seconds` so several samples exist. CPU package power via Intel RAPL energy counters when
    readable (Linux, device wide). Everything that cannot be measured is reported as None with a reason.
"""


import os
import resource
import sys
import threading
import time
from pathlib import Path

import numpy as np
import psutil
import torch

In [ ]:
def _nvml():
    try:
        import pynvml

        pynvml.nvmlInit()
        return pynvml, pynvml.nvmlDeviceGetHandleByIndex(0)
    except Exception:
        return None, None

In [ ]:
class _RAPL:
    """Intel RAPL package energy counter (Linux). Returns None when not readable."""

    def __init__(self):
        self.paths = []
        base = Path("/sys/class/powercap")
        if base.exists():
            for d in sorted(base.glob("intel-rapl:*")):
                p = d / "energy_uj"
                try:
                    p.read_text()
                    self.paths.append(p)
                except Exception:
                    pass

    @property
    def available(self) -> bool:
        return bool(self.paths)

    def read_j(self) -> float:
        return sum(int(p.read_text()) for p in self.paths) / 1e6

In [ ]:
class ResourceSampler(threading.Thread):
    def __init__(self, device: torch.device, interval: float = 0.05):
        super().__init__(daemon=True)
        self.interval = interval
        self.device = device
        self.proc = psutil.Process(os.getpid())
        self.samples: list[dict] = []
        self._stop_event = threading.Event()
        self.nvml, self.handle = _nvml() if device.type == "cuda" else (None, None)
        self.rapl = _RAPL()
        self.t_origin = time.perf_counter()
        self.e_origin = self.rapl.read_j() if self.rapl.available else None
        self.ready = threading.Event()

    def sample(self):
        s = {"t": time.perf_counter() - self.t_origin,
             "proc_cpu": self.proc.cpu_percent(None), "sys_cpu": psutil.cpu_percent(None),
             "rss": self.proc.memory_info().rss}
        if self.handle is not None:
            try:
                s["gpu_util"] = self.nvml.nvmlDeviceGetUtilizationRates(self.handle).gpu
                s["gpu_power_w"] = self.nvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
            except Exception:
                pass
        if self.e_origin is not None:
            s["rapl_j"] = self.rapl.read_j() - self.e_origin
        self.samples.append(s)
        return s

    def run(self):
        self.proc.cpu_percent(None)
        psutil.cpu_percent(None)
        self.ready.set()
        while not self._stop_event.wait(self.interval):
            self.sample()

    def stop(self):
        self._stop_event.set()
        self.join()
        self.sample()

In [ ]:
def _integrate_power(samples: list[dict], key: str) -> float | None:
    pts = [(s["t"], s[key]) for s in samples if key in s]
    if len(pts) < 2:
        return None
    t = np.array([p[0] for p in pts])
    p = np.array([p[1] for p in pts])
    return float(np.trapezoid(p, t)) if hasattr(np, "trapezoid") else float(np.trapz(p, t))

In [ ]:
@torch.no_grad()
def profile_forward(model, make_input, device: torch.device, warmup: int = 10, iters: int = 30,
                    min_power_seconds: float = 3.0, sample_interval: float = 0.05, tokens_per_item: int = 1,
                    batch_size: int = 1, macs: int | None = None, profiler_iters: int = 5,
                    profiler_table_path: str | Path | None = None) -> dict:
    """Run the full profiling protocol on `model(*make_input())`. Returns a flat metrics dict."""
    model.eval()
    notes = {}
    inp = make_input()

    def run_once():
        return model(*inp) if isinstance(inp, tuple) else model(inp)

    # ---- latency (no sampler, no profiler) ----
    for _ in range(warmup):
        run_once()
    synchronize(device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    lat = []
    for _ in range(iters):
        synchronize(device)
        t0 = time.perf_counter()
        run_once()
        synchronize(device)
        lat.append((time.perf_counter() - t0) * 1000.0)
    lat = np.array(lat)
    mean_ms = float(lat.mean())
    out = {
        "warmup_passes": warmup, "measured_passes": iters, "batch_size": batch_size,
        "latency_ms_mean": mean_ms, "latency_ms_std": float(lat.std(ddof=1)), "latency_ms_min": float(lat.min()),
        "latency_ms_median": float(np.median(lat)),
        "items_per_s": batch_size * 1000.0 / mean_ms, "tokens_per_s": batch_size * tokens_per_item * 1000.0 / mean_ms,
    }
    # ---- memory ----
    if device.type == "cuda":
        out["peak_gpu_allocated_mb"] = torch.cuda.max_memory_allocated(device) / 2**20
        out["peak_gpu_reserved_mb"] = torch.cuda.max_memory_reserved(device) / 2**20
        out["gpu_total_memory_mb"] = torch.cuda.get_device_properties(device).total_memory / 2**20
    elif device.type == "mps":
        out["peak_gpu_allocated_mb"] = torch.mps.driver_allocated_memory() / 2**20
        out["gpu_current_allocated_mb"] = torch.mps.current_allocated_memory() / 2**20
        notes["gpu_memory"] = "MPS reports driver-allocated memory (unified memory shared with the CPU), not a true peak"
    else:
        out["peak_gpu_allocated_mb"] = None
        notes["gpu_memory"] = "no GPU device"
    out["memory_scope"] = "single deployed model on GPU. CPU RSS is total notebook process including dataset caches"
    maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    out["peak_process_rss_mb_ru_maxrss"] = maxrss / 2**20 if sys.platform == "darwin" else maxrss / 2**10

    # ---- utilization + power: sampled repeated-pass loop, extended to several samples ----
    sampler = ResourceSampler(device, sample_interval)
    sampler.start()
    sampler.ready.wait()
    synchronize(device)
    first_sample = sampler.sample()
    t0 = time.perf_counter()
    k = 0
    while k < iters or (time.perf_counter() - t0) < min_power_seconds:
        run_once()
        k += 1
    synchronize(device)
    elapsed = time.perf_counter() - t0
    final_sample = sampler.sample()
    sampler.stop()
    s = sorted((v for v in sampler.samples if first_sample["t"] <= v["t"] <= final_sample["t"]), key=lambda v: v["t"])
    out["power_loop_passes"] = k
    out["power_loop_seconds"] = elapsed
    out["power_samples"] = len(s)
    out["sample_interval_s"] = sample_interval
    out["avg_process_cpu_percent"] = float(np.mean([x["proc_cpu"] for x in s])) if s else None
    out["avg_system_cpu_percent"] = float(np.mean([x["sys_cpu"] for x in s])) if s else None
    out["peak_process_rss_mb_sampled"] = max(x["rss"] for x in s) / 2**20 if s else None
    gpu_util = [x["gpu_util"] for x in s if "gpu_util" in x]
    out["avg_gpu_util_percent"] = float(np.mean(gpu_util)) if gpu_util else None
    gpu_e = _integrate_power(s, "gpu_power_w")
    if gpu_e is not None:
        out["avg_gpu_power_w"] = float(np.mean([x["gpu_power_w"] for x in s if "gpu_power_w" in x]))
        out["gpu_energy_per_pass_mj"] = 1000.0 * gpu_e / k
        notes["gpu_power"] = f"NVML board power, device wide (not per process), sampled every {sample_interval}s"
    else:
        out["avg_gpu_power_w"] = None
        out["gpu_energy_per_pass_mj"] = None
        notes["gpu_power"] = ("no NVML GPU power sensor on this platform (Apple MPS exposes no per-process or GPU power "
                              "counters without root powermetrics. CPU-only runs have no GPU)" if device.type != "cuda"
                              else "NVML unavailable (pynvml import or sensor failure)")
    rapl = [x["rapl_j"] for x in s if "rapl_j" in x]
    if len(rapl) >= 2:
        cpu_e = rapl[-1] - rapl[0]
        dt = s[-1]["t"] - s[0]["t"]
        out["avg_cpu_power_w"] = cpu_e / dt if dt > 0 else None
        out["cpu_energy_per_pass_mj"] = 1000.0 * cpu_e / k
        notes["cpu_power"] = "Intel RAPL package energy counter (/sys/class/powercap), whole socket, not per process"
    else:
        out["avg_cpu_power_w"] = None
        out["cpu_energy_per_pass_mj"] = None
        notes["cpu_power"] = ("CPU power sensor unavailable: RAPL counters not readable (containers/Colab/ARM) and macOS "
                              "exposes CPU power only through root-only powermetrics")
    if device.type != "cuda":
        out["avg_gpu_util_percent"] = None
        notes["gpu_util"] = "GPU utilization counters not exposed on this platform"
    # ---- energy per MAC / FLOP ----
    if macs:
        out["macs"] = macs
        out["flops"] = 2 * macs
        for key, label in (("gpu_energy_per_pass_mj", "gpu"), ("cpu_energy_per_pass_mj", "cpu")):
            e = out.get(key)
            out[f"{label}_energy_per_mac_pj"] = (e * 1e9 / macs) if e is not None else None      # mJ -> pJ
            out[f"{label}_energy_per_flop_pj"] = (e * 1e9 / (2 * macs)) if e is not None else None

    # ---- torch.profiler (operator breakdown + FLOP estimate), separate from the timing above ----
    try:
        from torch.profiler import ProfilerActivity, profile

        acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
        with profile(activities=acts, with_flops=True, profile_memory=True) as prof:
            for _ in range(profiler_iters):
                run_once()
            synchronize(device)
        ka = prof.key_averages()
        total_flops = sum(getattr(e, "flops", 0) or 0 for e in ka)
        out["profiler_flops_per_pass"] = total_flops / profiler_iters if total_flops else None
        sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
        try:
            table = ka.table(sort_by=sort_key, row_limit=15)
        except Exception:
            table = ka.table(sort_by="cpu_time_total", row_limit=15)
        if profiler_table_path:
            Path(profiler_table_path).parent.mkdir(parents=True, exist_ok=True)
            Path(profiler_table_path).write_text(table)
        out["profiler_table_head"] = "\n".join(table.splitlines()[:8])
    except Exception as e:  # profiler not supported on some backends
        out["profiler_flops_per_pass"] = None
        notes["profiler"] = f"torch.profiler failed: {e}"
    out["notes"] = notes
    return out

In [ ]:
"""Quantization-aware training for ResNet-18 on CIFAR-10 (Task 3) with self-implemented fake quantization + STE.

Fake quantization (forward) for every Conv2d / Linear:
    weights     : symmetric per-output-channel, b_w bits, scale recomputed from the FP32 master weights each step
    activations : asymmetric per-tensor unsigned 8 bit, range [x_min, x_max] from an observer
    bias        : FP32, untouched
Straight-through estimator (backward): d(fake_quant(x))/dx = 1 inside the clipping range, 0 outside.
Implemented as  x_c = clamp(x, lo, hi), out = x_c + (round_dequant(x_c) - x_c).detach()
so autograd sees the identity for the rounding step and clamp's own zero gradient outside the range.

Observers: initialised with exact min/max over `init_batches` training batches (no fake quant yet), then updated
with an exponential moving average (momentum `ema_momentum`) during training, frozen after `freeze_after_epoch`
epochs and for the final evaluation.
BatchNorm: kept as separate FP32 layers in training mode during QAT (batch statistics) and NOT folded into the
convolutions. The fake-quantized weight is therefore the convolution weight, and the BN affine transform runs in
floating point after the convolution. (Folding at deployment would change the effective weights and would
require re-quantizing them, so it is deliberately not done here. This is stated in the report.)
"""


import copy
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F




HF_REPO = "edadaltocg/resnet18_cifar10"

In [ ]:
def load_resnet18_cifar10(device: torch.device):
    """timm ResNet-18 with the CIFAR stem (3x3 stride-1 conv1, no max-pool) and the pretrained weights from
    huggingface.co/edadaltocg/resnet18_cifar10 (the checkpoint of the edadaltocg/detectors repository)."""
    import timm
    from huggingface_hub import hf_hub_download

    m = timm.create_model("resnet18", num_classes=10, pretrained=False)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    path = hf_hub_download(HF_REPO, "pytorch_model.bin")
    sd = torch.load(path, map_location="cpu")
    m.load_state_dict(sd, strict=True)
    return m.to(device).eval(), path

In [ ]:
class _FakeQuantBase(nn.Module):
    def __init__(self, w_bits: int, a_bits: int, ema_momentum: float = 0.9):
        super().__init__()
        self.w_bits, self.a_bits = w_bits, a_bits
        self.ema_momentum = ema_momentum
        self.register_buffer("x_min", torch.tensor(0.0))
        self.register_buffer("x_max", torch.tensor(0.0))
        self.register_buffer("initialized", torch.tensor(False))
        self.mode = "fp"  # 'fp' | 'observe' | 'train' | 'frozen'
        self.record_errors = False
        self.err_acc = None
        self.cache_weights = False   # inference only: fake-quantized weight computed once and reused
        self._w_cache = None

    def _observe(self, x: torch.Tensor) -> None:
        mn, mx = x.detach().min(), x.detach().max()
        if not bool(self.initialized):
            self.x_min, self.x_max = mn, mx
            self.initialized.fill_(True)
        elif self.mode == "observe":  # exact aggregation during initialisation
            self.x_min, self.x_max = torch.minimum(self.x_min, mn), torch.maximum(self.x_max, mx)
        else:  # EMA during training
            m = self.ema_momentum
            self.x_min = m * self.x_min + (1 - m) * mn
            self.x_max = m * self.x_max + (1 - m) * mx

    def quant_input(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == "fp":
            return x
        if self.mode in ("observe", "train"):
            self._observe(x)
        if self.mode == "observe":
            return x
        xq = fake_quant_activation(x, self.x_min, self.x_max, self.a_bits)
        if self.record_errors:
            self._acc("X", x, xq)
        return xq

    def quant_weight(self, w: torch.Tensor) -> torch.Tensor:
        if self.mode in ("fp", "observe"):
            return w
        if self.cache_weights and not self.record_errors and not torch.is_grad_enabled():
            if self._w_cache is None:
                self._w_cache = fake_quant_weight(w.detach(), self.w_bits, channel_dim=0)
            return self._w_cache
        wq = fake_quant_weight(w, self.w_bits, channel_dim=0)
        if self.record_errors:
            self._acc("W", w, wq)
        return wq

    def _acc(self, key: str, a: torch.Tensor, a_hat: torch.Tensor) -> None:
        if self.err_acc is None:
            self.err_acc = {"X_num": 0.0, "X_den": 0.0, "W_num": 0.0, "W_den": 0.0, "clip": 0, "count": 0}
        self.err_acc[f"{key}_num"] += float(((a.detach() - a_hat.detach()) ** 2).sum())
        self.err_acc[f"{key}_den"] += float((a.detach() ** 2).sum())
        if key == "X":
            self.err_acc["clip"] += int(((a < self.x_min) | (a > self.x_max)).sum())
            self.err_acc["count"] += a.numel()

In [ ]:
class FakeQuantConv2d(_FakeQuantBase):
    def __init__(self, conv: nn.Conv2d, w_bits: int, a_bits: int = 8, **kw):
        super().__init__(w_bits, a_bits, **kw)
        self.conv = conv

    def forward(self, x):
        x = self.quant_input(x)
        w = self.quant_weight(self.conv.weight)
        c = self.conv
        return F.conv2d(x, w, c.bias, c.stride, c.padding, c.dilation, c.groups)

In [ ]:
class FakeQuantLinear(_FakeQuantBase):
    def __init__(self, lin: nn.Linear, w_bits: int, a_bits: int = 8, **kw):
        super().__init__(w_bits, a_bits, **kw)
        self.lin = lin

    def forward(self, x):
        x = self.quant_input(x)
        w = self.quant_weight(self.lin.weight)
        return F.linear(x, w, self.lin.bias)

In [ ]:
def prepare_qat(model: nn.Module, w_bits: int, a_bits: int = 8, ema_momentum: float = 0.9) -> nn.Module:
    """Deep copy and wrap every Conv2d and Linear with a fake-quant layer (weights + input activations)."""
    model = copy.deepcopy(model)

    def _wrap(mod: nn.Module):
        for name, child in list(mod.named_children()):
            if isinstance(child, nn.Conv2d):
                setattr(mod, name, FakeQuantConv2d(child, w_bits, a_bits, ema_momentum=ema_momentum))
            elif isinstance(child, nn.Linear):
                setattr(mod, name, FakeQuantLinear(child, w_bits, a_bits, ema_momentum=ema_momentum))
            else:
                _wrap(child)

    _wrap(model)
    return model

In [ ]:
def fq_modules(model: nn.Module) -> dict[str, _FakeQuantBase]:
    return {n: m for n, m in model.named_modules() if isinstance(m, _FakeQuantBase)}

In [ ]:
def set_mode(model: nn.Module, mode: str) -> None:
    for m in fq_modules(model).values():
        m.mode = mode

In [ ]:
def freeze_for_inference(model: nn.Module) -> None:
    """Final deployment state: observers frozen and the fake-quantized weights computed once (the master weights no longer
    change), so inference pays only the activation quantize/dequantize in front of each FP32 conv/linear kernel."""
    for m in fq_modules(model).values():
        m.mode = "frozen"
        m.cache_weights = True
        m._w_cache = None

In [ ]:
@torch.no_grad()
def init_observers(model: nn.Module, loader, device: torch.device, n_batches: int = 20) -> None:
    model.eval()
    set_mode(model, "observe")
    for i, (x, _) in enumerate(loader):
        if i >= n_batches:
            break
        model(x.to(device))
    set_mode(model, "train")

In [ ]:
@torch.no_grad()
def evaluate_classifier(model: nn.Module, loader, device: torch.device, with_f1: bool = True) -> dict:
    model.eval()
    loss_sum, n, top1, top5 = 0.0, 0, 0, 0
    preds, labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += float(F.cross_entropy(logits, y, reduction="sum"))
        t5 = logits.topk(5, dim=1).indices
        top1 += int((t5[:, 0] == y).sum())
        top5 += int((t5 == y.unsqueeze(1)).any(dim=1).sum())
        n += y.numel()
        preds.append(t5[:, 0].cpu())
        labels.append(y.cpu())
    out = {"loss": loss_sum / n, "top1": 100.0 * top1 / n, "top5": 100.0 * top5 / n, "n": n}
    if with_f1:
        from sklearn.metrics import f1_score

        out["macro_f1"] = 100.0 * f1_score(torch.cat(labels).numpy(), torch.cat(preds).numpy(), average="macro")
    return out

In [ ]:
def train_qat(model: nn.Module, loaders, device: torch.device, epochs: int = 8, lr: float = 0.01, momentum: float = 0.9,
              weight_decay: float = 5e-4, freeze_after_epoch: int | None = None, init_batches: int = 20,
              seed: int = 624, log=print) -> tuple[nn.Module, dict]:
    """Fine-tune with fake quantization. Checkpoint selection uses validation top-1 only. Returns (best model, history)."""
    torch.manual_seed(seed)
    if freeze_after_epoch is None:
        freeze_after_epoch = max(1, epochs - 1)   # observers frozen for the final epoch
    init_observers(model, loaders["train"], device, init_batches)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": [], "epoch_seconds": [],
            "observer_frozen": []}
    best_acc, best_state, best_epoch = -1.0, None, -1
    t_start = time.perf_counter()
    for ep in range(1, epochs + 1):
        frozen = ep > freeze_after_epoch
        set_mode(model, "frozen" if frozen else "train")
        model.train()
        t0 = time.perf_counter()
        loss_sum, correct, n = 0.0, 0, 0
        for x, y in loaders["train"]:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            loss_sum += loss.item() * y.numel()
            correct += int((logits.argmax(1) == y).sum())
            n += y.numel()
        sched.step()
        set_mode(model, "frozen")  # evaluation never updates observers
        val = evaluate_classifier(model, loaders["val"], device, with_f1=False)
        set_mode(model, "frozen" if frozen else "train")
        hist["epoch"].append(ep)
        hist["train_loss"].append(loss_sum / n)
        hist["train_acc"].append(100.0 * correct / n)
        hist["val_loss"].append(val["loss"])
        hist["val_acc"].append(val["top1"])
        hist["lr"].append(opt.param_groups[0]["lr"])
        hist["epoch_seconds"].append(time.perf_counter() - t0)
        hist["observer_frozen"].append(frozen)
        if val["top1"] > best_acc:
            best_acc, best_epoch = val["top1"], ep
            best_state = copy.deepcopy(model.state_dict())
        log(f"epoch {ep}/{epochs} train_loss={loss_sum / n:.4f} train_acc={100.0 * correct / n:.2f} "
            f"val_loss={val['loss']:.4f} val_acc={val['top1']:.2f} frozen={frozen} ({hist['epoch_seconds'][-1]:.0f}s)")
    model.load_state_dict(best_state)
    set_mode(model, "frozen")
    hist["best_epoch"] = best_epoch
    hist["best_val_acc"] = best_acc
    hist["total_train_seconds"] = time.perf_counter() - t_start
    hist["config"] = {"epochs": epochs, "lr": lr, "momentum": momentum, "weight_decay": weight_decay, "nesterov": True,
                      "scheduler": "cosine", "freeze_after_epoch": freeze_after_epoch, "init_batches": init_batches,
                      "seed": seed, "optimizer": "SGD"}
    return model, hist

In [ ]:
@torch.no_grad()
def qat_layer_errors(model: nn.Module, loader, device: torch.device, n_batches: int = 4) -> dict[str, dict]:
    """Normalized weight error (fake-quant weight vs FP32 master weight) and activation error on validation inputs."""
    mods = fq_modules(model)
    for m in mods.values():
        m.record_errors, m.err_acc = True, None
    model.eval()
    set_mode(model, "frozen")
    for i, (x, _) in enumerate(loader):
        if i >= n_batches:
            break
        model(x.to(device))
    out = {}
    for n, m in mods.items():
        a = m.err_acc
        out[n] = {"E_W": a["W_num"] / (a["W_den"] + 1e-12), "E_X": a["X_num"] / (a["X_den"] + 1e-12),
                  "clip_frac": a["clip"] / max(a["count"], 1)}
        m.record_errors, m.err_acc = False, None
    return out

In [ ]:
def export_qat_state_dict(model: nn.Module) -> tuple[dict[str, torch.Tensor], dict]:
    """Integer export of a QAT model: int8 or packed int4 weights, FP32 per-channel scales, activation scale and
    zero point per layer, FP32 biases and BatchNorm parameters."""
    state, meta = {}, {}
    for n, m in model.named_modules():
        if isinstance(m, _FakeQuantBase):
            w = m.conv.weight if isinstance(m, FakeQuantConv2d) else m.lin.weight
            q, s, _ = quantize_weight(w.detach(), m.w_bits, channel_dim=0)
            key = f"{n}.q_weight_packed" if m.w_bits <= 4 else f"{n}.q_weight"
            state[key] = pack_int4(q.cpu()) if m.w_bits <= 4 else q.cpu()
            state[f"{n}.w_scale"] = s.cpu()
            sc, z = activation_qparams(m.x_min, m.x_max, m.a_bits)
            state[f"{n}.a_scale"] = sc.cpu()
            state[f"{n}.a_zero"] = z.cpu().to(torch.uint8)
            b = m.conv.bias if isinstance(m, FakeQuantConv2d) else m.lin.bias
            if b is not None:
                state[f"{n}.bias"] = b.detach().cpu()
            meta[n] = {"w_bits": m.w_bits, "a_bits": m.a_bits, "packed": m.w_bits <= 4, "shape": list(w.shape)}
    for n, t in model.state_dict().items():
        if any(n.startswith(k + ".") for k in meta):
            continue
        state[n] = t.detach().cpu()
    return state, meta

In [ ]:
def theoretical_packed_bytes_qat(model: nn.Module) -> dict:
    qbits, qparam, other = 0, 0, 0
    names = set()
    for n, m in model.named_modules():
        if isinstance(m, _FakeQuantBase):
            names.add(n)
            w = m.conv.weight if isinstance(m, FakeQuantConv2d) else m.lin.weight
            qbits += w.numel() * m.w_bits
            qparam += w.shape[0] * 4 + 8
            b = m.conv.bias if isinstance(m, FakeQuantConv2d) else m.lin.bias
            other += b.numel() * 4 if b is not None else 0
    for n, t in model.state_dict().items():
        if any(n.startswith(k + ".") for k in names):
            continue
        other += t.numel() * t.element_size()
    return {"quantized_weight_bytes": qbits / 8, "qparam_bytes": qparam, "fp32_other_bytes": other,
            "total_bytes": qbits / 8 + qparam + other}

In [ ]:
@torch.no_grad()
def resnet_mac_breakdown(model: nn.Module, device: torch.device, input_shape=(1, 3, 32, 32)) -> dict:
    """MACs per forward pass counted by hooks on every Conv2d and Linear (1 MAC = 2 FLOPs). BN, ReLU, pooling and
    residual adds are elementwise and listed separately as an estimate."""
    counts = {"conv": 0, "linear": 0}
    elementwise = [0]
    handles = []

    def conv_hook(m, i, o):
        counts["conv"] += int(o.numel() // o.shape[0] * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1] * o.shape[0])

    def lin_hook(m, i, o):
        counts["linear"] += int(o.numel() * m.in_features)

    def elem_hook(m, i, o):
        elementwise[0] += int(o.numel() * 2)

    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            handles.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            handles.append(mod.register_forward_hook(lin_hook))
        elif isinstance(mod, (nn.BatchNorm2d, nn.ReLU)):
            handles.append(mod.register_forward_hook(elem_hook))
    model.eval()
    model(torch.zeros(*input_shape, device=device))
    for h in handles:
        h.remove()
    macs = counts["conv"] + counts["linear"]
    return {"input_shape": list(input_shape), "macs_breakdown": counts, "macs_total": macs, "flops_total": 2 * macs,
            "other_elementwise_flops_estimate": elementwise[0],
            "convention": "1 MAC = 2 FLOPs. Conv and linear MACs only. BN/ReLU/pool/residual listed separately"}

In [ ]:
def plot_training_curves(histories: dict[str, dict], path: str | Path) -> str:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for label, h in histories.items():
        axes[0].plot(h["epoch"], h["train_loss"], "-o", ms=3, label=f"{label} train")
        axes[0].plot(h["epoch"], h["val_loss"], "--s", ms=3, label=f"{label} val")
        axes[1].plot(h["epoch"], h["train_acc"], "-o", ms=3, label=f"{label} train")
        axes[1].plot(h["epoch"], h["val_acc"], "--s", ms=3, label=f"{label} val")
    axes[0].set_title("loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend(fontsize=7)
    axes[1].set_title("top-1 accuracy (%)")
    axes[1].set_xlabel("epoch")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return str(path)

In [ ]:
"""High-level experiment runners used by the notebook and the CLI scripts. Every runner writes JSON to the results
directory so the notebook can be re-executed stage by stage."""


import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch














CONFIGS = [QuantSpec(8, 8, "static"), QuantSpec(8, 8, "dynamic"), QuantSpec(4, 8, "static"), QuantSpec(4, 8, "dynamic")]
ALPHAS = (0.25, 0.5, 0.75)

In [ ]:
@dataclass
class Config:
    results_dir: str = "results"
    device: str | None = None               # None = auto (cuda > mps > cpu)
    seed: int = DEFAULT_SEED
    calib_seed: int = 624
    n_calib: int = N_CALIB
    eval_batch_size: int = 16               # evaluation and calibration batch size (fixed across methods)
    profile_batch_size: int = 1
    warmup: int = 10
    iters: int = 30
    min_power_seconds: float = 3.0
    test_blocks: int | None = None          # None = full test split
    val_blocks: int | None = None           # None = full validation split (alpha selection)
    analysis_blocks: int = 16               # fixed validation blocks for errors / distributions
    stat_layers: tuple = (0, 5, 11)
    gptq_block_size: int = 128
    gptq_percdamp: float = 0.01
    gptq_act_order: bool = False
    reuse: bool = True                      # reuse saved JSON / GPTQ weights when present

    @property
    def root(self) -> Path:
        return Path(self.results_dir)

    @property
    def figures(self) -> Path:
        p = self.root / "figures"
        p.mkdir(parents=True, exist_ok=True)
        return p

In [ ]:
class Context:
    """Base model, tokenizer, data splits and calibration set shared by all GPT-2 experiments."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        set_seed(cfg.seed)
        self.device = select_device(cfg.device)
        self.model, self.tok = load_gpt2(self.device)
        self.splits = load_wikitext_blocks(self.tok)
        self.calib, self.calib_idx = sample_calibration(self.splits["train"], cfg.n_calib, cfg.calib_seed)
        self.test = self.splits["test"].blocks if cfg.test_blocks is None else self.splits["test"].blocks[: cfg.test_blocks]
        self.val = self.splits["validation"].blocks if cfg.val_blocks is None else self.splits["validation"].blocks[: cfg.val_blocks]
        self.analysis = self.splits["validation"].blocks[: cfg.analysis_blocks]
        self.profile_input = self.test[:1].to(self.device)   # one fixed 128-token sequence, batch size 1
        self.macs = gpt2_mac_breakdown(self.model, SEQ_LEN, cfg.profile_batch_size)
        self.env = environment_info(self.device)

    def data_summary(self) -> dict:
        return {k: v.summary() for k, v in self.splits.items()} | {
            "calibration": {"n": int(self.calib.shape[0]), "seed": self.cfg.calib_seed, "source": "train blocks",
                            "indices_head": self.calib_idx[:10].tolist(), "indices": self.calib_idx.tolist()},
            "eval_batch_size": self.cfg.eval_batch_size, "test_blocks_used": int(self.test.shape[0]),
            "val_blocks_used": int(self.val.shape[0]), "analysis_blocks": int(self.analysis.shape[0])}

    def profile(self, model, tag: str, extra: dict | None = None) -> dict:
        model.eval()
        inp = self.profile_input

        def make_input():
            return ({"input_ids": inp, "use_cache": False},)

        # model(**kwargs) wrapper so profile_forward can call model(*inp)
        class _Call:
            def __init__(self, m):
                self.m = m

            def eval(self):
                self.m.eval()
                return self

            def __call__(self, kw):
                return self.m(**kw)

        import gc
        offload_reference = model is not self.model
        if offload_reference:
            self.model.cpu()
        gc.collect()
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
        p = profile_forward(_Call(model), make_input, self.device, self.cfg.warmup, self.cfg.iters,
                            self.cfg.min_power_seconds, tokens_per_item=SEQ_LEN, batch_size=self.cfg.profile_batch_size,
                            macs=self.macs["macs_total"], profiler_table_path=self.cfg.root / "profiler" / f"{tag}.txt")
        if offload_reference:
            self.model.to(self.device)
        if extra:
            p.update(extra)
        return p

In [ ]:
def storage_report(model, baseline_fp32_bytes: int | None = None) -> dict:
    """Actual serialized sizes: integer export (packed int4 when W4), simulated FP32 runtime copy, theoretical packed."""
    export, meta = export_quantized_state_dict(model)
    sd = model.state_dict()
    runtime = {k: v for k, v in sd.items() if not k.endswith(".q_weight")}
    theo = theoretical_packed_bytes(model)
    any_packed = any(l["packed"] for l in meta["layers"].values()) if meta["layers"] else False
    return {"integer_export_mb": state_dict_bytes({"state_dict": export, "metadata": meta}) / 2**20,
            "simulated_runtime_fp32_mb": state_dict_bytes(sd) / 2**20,
            "fp32_tensor_subset_mb": state_dict_bytes(runtime) / 2**20,
            "theoretical_packed_mb": theo["total_bytes"] / 2**20,
            "int4_packed": any_packed,
            "export_includes": "int8 / packed-int4 weights, FP32 per-channel scales, static A8 scale+zero point, FP32 biases, "
                               "all unquantized FP32 tensors (embeddings tied once, LayerNorm), runtime smoothing vectors and format metadata"}

In [ ]:
def run_task1(ctx: Context) -> dict:
    cfg = ctx.cfg
    out_path = cfg.root / "task1_baseline.json"
    if cfg.reuse and out_path.exists():
        return load_json(out_path)
    m = ctx.model
    res = {"architecture": architecture_summary(m), "data": ctx.data_summary(), "environment": ctx.env,
           "macs": ctx.macs, "unique_parameters": unique_parameter_count(m),
           "all_parameter_entries": sum(p.numel() for p in m.parameters()),
           "serialized_fp32_mb": state_dict_bytes({k: v for k, v in m.state_dict().items() if k != "lm_head.weight"}) / 2**20,
           "serialized_fp32_mb_with_tied_copy": state_dict_bytes(m.state_dict()) / 2**20}
    with Timer() as t:
        res["lm"] = evaluate_lm(m, ctx.test, cfg.eval_batch_size, ctx.device)
    res["lm"]["eval_seconds"] = t.seconds
    res["profile"] = ctx.profile(m, "fp32_baseline")
    res["execution"] = "FP32 PyTorch kernels"
    save_json(res, out_path)
    return res

In [ ]:
def evaluate_config(ctx: Context, q_model, spec: QuantSpec, method: str, extra: dict | None = None,
                    calibrate: bool = True) -> dict:
    cfg = ctx.cfg
    q_model.to(ctx.device).eval()
    tag = f"{method}_{spec.name}"
    res = {"method": method, "config": spec.name, "w_bits": spec.w_bits, "a_bits": spec.a_bits, "act_mode": spec.act_mode,
           "execution": "simulated quantization (quantize-dequantize around FP32 matmul kernels)"}
    if spec.act_mode == "static" and calibrate:
        with Timer() as t:
            calibrate_static_ranges(q_model, ctx.calib.to(ctx.device), cfg.eval_batch_size, ctx.device)
        res["calibration_seconds"] = t.seconds
    with Timer() as t:
        res["lm"] = evaluate_lm(q_model, ctx.test, cfg.eval_batch_size, ctx.device)
    res["lm"]["eval_seconds"] = t.seconds
    res["storage"] = storage_report(q_model)
    res["profile"] = ctx.profile(q_model, tag)
    errs = layer_errors(ctx.model, q_model, ctx.analysis, cfg.eval_batch_size, ctx.device)
    res["layer_errors"] = errs
    res["error_summary"] = summarize_layer_errors(errs)
    if extra:
        res.update(extra)
    return res

In [ ]:
def run_linear_ptq(ctx: Context) -> dict:
    cfg = ctx.cfg
    out_path = cfg.root / "task2_1_linear_ptq.json"
    if cfg.reuse and out_path.exists():
        return load_json(out_path)
    results = {}
    for spec in CONFIGS:
        print(f"[linear PTQ] {spec.name}")
        q_model = quantize_gpt2(ctx.model, spec)
        results[spec.name] = evaluate_config(ctx, q_model, spec, "linear_ptq")
        for lid in cfg.stat_layers:
            for projection in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
                results[spec.name].setdefault("figures", []).append(
                    plot_quant_vs_original(q_model, ctx.model, ctx.analysis, ctx.device, f"transformer.h.{lid}.{projection}",
                                           cfg.figures, f"linear_{spec.name}"))
        del q_model
    save_json(results, out_path)
    return results

In [ ]:
def run_distribution_stats(ctx: Context) -> dict:
    """Original-model statistics for blocks 0, 5, 11 on the analysis validation blocks (used by 2.2 and 2.3)."""
    cfg = ctx.cfg
    path = cfg.root / "stats_original.npz.json"
    stats = collect_projection_stats(ctx.model, ctx.analysis, cfg.eval_batch_size, ctx.device, cfg.stat_layers)
    save_json({"table": stats_table(stats)}, path)
    return stats

In [ ]:
def run_smoothquant(ctx: Context, stats_original: dict | None = None) -> dict:
    cfg = ctx.cfg
    out_path = cfg.root / "task2_2_smoothquant.json"
    if cfg.reuse and out_path.exists():
        return load_json(out_path)
    stats_original = stats_original or run_distribution_stats(ctx)
    res = {"stats_original": stats_table(stats_original), "alphas": list(ALPHAS), "equivalence": {}, "alpha_search": {},
           "selected_alpha": {}, "configs": {}, "figures": []}
    act_max = collect_channel_maxima(ctx.model, ctx.calib.to(ctx.device), cfg.eval_batch_size, ctx.device)

    def smoothed_for(a: float):
        scales = compute_smooth_scales(ctx.model, act_max, a)
        return scales, apply_smoothing(ctx.model, scales)

    # alpha selection on validation perplexity, per configuration (one smoothed model alive at a time)
    search = {spec.name: {} for spec in CONFIGS}
    for a in ALPHAS:
        scales, sm = smoothed_for(a)
        res["equivalence"][str(a)] = verify_equivalence(ctx.model, sm, ctx.analysis, ctx.device)
        print(f"[SmoothQuant] alpha={a} equivalence: {res['equivalence'][str(a)]}")
        for spec in CONFIGS:
            q = quantize_smoothed(sm, ctx.model, scales, spec)
            if spec.act_mode == "static":
                calibrate_static_ranges(q, ctx.calib.to(ctx.device), cfg.eval_batch_size, ctx.device)
            v = evaluate_lm(q, ctx.val, cfg.eval_batch_size, ctx.device)
            search[spec.name][str(a)] = v
            print(f"[SmoothQuant] {spec.name} alpha={a} val ppl={v['ppl']:.3f}")
            del q
        del sm
    for spec in CONFIGS:
        best = min(search[spec.name], key=lambda k: search[spec.name][k]["ppl"])
        res["alpha_search"][spec.name] = search[spec.name]
        res["selected_alpha"][spec.name] = float(best)
    # final evaluation with the selected alpha
    stats_after = {}
    cache = {}
    for spec in CONFIGS:
        a = res["selected_alpha"][spec.name]
        if a not in cache:
            cache[a] = smoothed_for(a)
        scales, sm = cache[a]
        q = quantize_smoothed(sm, ctx.model, scales, spec).to(ctx.device)
        sm.cpu()
        for k in scales:
            scales[k] = scales[k].cpu()
        r = evaluate_config(ctx, q, spec, "smoothquant", extra={"alpha": a})
        r["scale_stats"] = {n: {"min": float(s.min()), "max": float(s.max()), "mean": float(s.mean())}
                            for n, s in scales.items() if any(f".h.{i}." in n for i in cfg.stat_layers)}
        res["configs"][spec.name] = r
        if a not in stats_after:
            sm.to(ctx.device)
            runtime_div = {n: m.in_scale for n, m in sm.named_modules() if isinstance(m, ScaledInputConv1D)}
            stats_after[a] = collect_projection_stats(sm, ctx.analysis, cfg.eval_batch_size, ctx.device, cfg.stat_layers,
                                                      input_transform=runtime_div)
            res[f"stats_smoothed_alpha_{a}"] = stats_table(stats_after[a])
            res["figures"] += plot_distributions({"original": stats_original, f"smoothed a={a}": stats_after[a]},
                                                 cfg.figures, f"smoothquant_a{a}")
            sm.cpu()
        for lid in cfg.stat_layers:
            for projection in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
                res["figures"].append(plot_quant_vs_original(q, ctx.model, ctx.analysis, ctx.device,
                                                             f"transformer.h.{lid}.{projection}", cfg.figures, f"sq_{spec.name}"))
        del q
    save_json(res, out_path)
    return res

In [ ]:
def run_gptq(ctx: Context, stats_original: dict | None = None) -> dict:
    cfg = ctx.cfg
    out_path = cfg.root / "task2_3_gptq.json"
    if cfg.reuse and out_path.exists():
        return load_json(out_path)
    stats_original = stats_original or run_distribution_stats(ctx)
    res = {"stats_original": stats_table(stats_original), "quantization": {}, "configs": {}, "figures": []}
    weights = {}
    for bits in (8, 4):
        wpath = cfg.root / f"gptq_w{bits}.pt"
        if cfg.reuse and wpath.exists():
            saved = torch.load(wpath)
            weights[bits], res["quantization"][f"W{bits}"] = saved["weights"], saved["info"]
            continue
        print(f"[GPTQ] quantizing W{bits}")
        import psutil, os
        rss0 = psutil.Process(os.getpid()).memory_info().rss
        qw, info = gptq_quantize_gpt2(ctx.model, ctx.calib, cfg.eval_batch_size, ctx.device, bits, cfg.gptq_block_size,
                                      cfg.gptq_percdamp, cfg.gptq_act_order)
        info["rss_increase_mb"] = (psutil.Process(os.getpid()).memory_info().rss - rss0) / 2**20
        weights[bits] = qw
        res["quantization"][f"W{bits}"] = info
        torch.save({"weights": qw, "info": info}, wpath)
    for spec in CONFIGS:
        print(f"[GPTQ] {spec.name}")
        q = build_gptq_model(ctx.model, weights[spec.w_bits], spec)
        res["configs"][spec.name] = evaluate_config(ctx, q, spec, "gptq")
        for lid in cfg.stat_layers:
            for projection in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
                res["figures"].append(plot_quant_vs_original(q, ctx.model, ctx.analysis, ctx.device,
                                                             f"transformer.h.{lid}.{projection}", cfg.figures, f"gptq_{spec.name}"))
        del q
    for k in ("W8", "W4"):
        res["quantization"][k]["layers"] = [{"name": l["name"], "seconds": l["seconds"], "damp": l["damp"],
                                             "cholesky_retries": l["cholesky_retries"], "dead_inputs": l["dead_inputs"]}
                                            for l in res["quantization"][k]["layers"]]
    save_json(res, out_path)
    return res

In [ ]:
def consolidated_rows(t1: dict, ptq: dict, sq: dict, gptq: dict) -> list[dict]:
    def row(name, lm, prof, size_mb, size_theo, exec_label, errs=None, alpha=None):
        return {"configuration": name, "accuracy_%": lm["accuracy"], "nll": lm["nll"], "ppl": lm["ppl"],
                "size_mb": size_mb, "size_theoretical_packed_mb": size_theo,
                "macs_G": prof.get("macs", t1["macs"]["macs_total"]) / 1e9, "flops_G": 2 * prof.get("macs", t1["macs"]["macs_total"]) / 1e9,
                "latency_ms": prof["latency_ms_mean"], "latency_std_ms": prof["latency_ms_std"],
                "seq_per_s": prof["items_per_s"], "tokens_per_s": prof["tokens_per_s"],
                "peak_gpu_mb": prof.get("peak_gpu_allocated_mb"), "peak_rss_mb": prof.get("peak_process_rss_mb_sampled"),
                "cpu_util_%": prof.get("avg_process_cpu_percent"), "gpu_util_%": prof.get("avg_gpu_util_percent"),
                "gpu_power_w": prof.get("avg_gpu_power_w"), "gpu_energy_mj": prof.get("gpu_energy_per_pass_mj"),
                "cpu_energy_mj": prof.get("cpu_energy_per_pass_mj"), "energy_per_mac_pj": prof.get("gpu_energy_per_mac_pj") or prof.get("cpu_energy_per_mac_pj"),
                "E_Y_mean": errs["E_Y"] if errs else None, "alpha": alpha, "execution": exec_label}

    rows = [row("FP32 baseline", t1["lm"], t1["profile"], t1["serialized_fp32_mb"], t1["serialized_fp32_mb"], "FP32 kernels")]
    for method, data in (("Linear PTQ", ptq), ("SmoothQuant", sq["configs"]), ("GPTQ", gptq["configs"])):
        for cname, r in data.items():
            rows.append(row(f"{method} {cname}", r["lm"], r["profile"], r["storage"]["integer_export_mb"],
                            r["storage"]["theoretical_packed_mb"], "simulated (fake-quant + FP32 matmul)",
                            r["error_summary"], r.get("alpha")))
    return rows

In [ ]:
@dataclass
class QATConfig:
    results_dir: str = "results"
    device: str | None = None
    seed: int = DEFAULT_SEED
    epochs: int = 8
    lr: float = 0.01
    batch_size: int = 128
    freeze_after_epoch: int | None = None
    init_batches: int = 20
    warmup: int = 10
    iters: int = 30
    min_power_seconds: float = 3.0
    data_root: str = "data"
    data_source: str = "torchvision"       # or "hf" (uoft-cs/cifar10 mirror)
    num_workers: int = 2
    train_subset: int | None = None        # smoke tests only
    reuse: bool = True

    @property
    def root(self) -> Path:
        return Path(self.results_dir)

In [ ]:
def run_task3(qcfg: QATConfig, log=print) -> dict:





    out_path = qcfg.root / "task3_qat.json"
    if qcfg.reuse and out_path.exists():
        return load_json(out_path)
    set_seed(qcfg.seed)
    device = select_device(qcfg.device)
    loaders, split = cifar10_loaders(qcfg.data_root, qcfg.batch_size, num_workers=qcfg.num_workers, seed=qcfg.seed,
                                     source=qcfg.data_source)
    split["source"] = qcfg.data_source
    if qcfg.train_subset:  # smoke test only
        from torch.utils.data import DataLoader, Subset
        for k in ("train", "train_eval", "val", "test"):
            ds = loaders[k].dataset
            loaders[k] = DataLoader(Subset(ds, list(range(min(qcfg.train_subset, len(ds))))), batch_size=qcfg.batch_size,
                                    shuffle=(k == "train"))
    fp_model, ckpt = load_resnet18_cifar10(device)
    macs = resnet_mac_breakdown(fp_model, device)
    res = {"checkpoint": ckpt, "split": split, "environment": environment_info(device), "macs": macs,
           "config": asdict(qcfg), "models": {}, "histories": {}}
    x0 = next(iter(loaders["test"]))[0][:1].to(device)

    def profile(model, tag):
        return profile_forward(model, lambda: x0, device, qcfg.warmup, qcfg.iters, qcfg.min_power_seconds, tokens_per_item=1,
                               batch_size=1, macs=macs["macs_total"], profiler_table_path=qcfg.root / "profiler" / f"{tag}.txt")

    def metrics(model, tag, exec_label):
        if tag != "fp32":
            freeze_for_inference(model)
        r = {"execution": exec_label,
             "train": evaluate_classifier(model, loaders["train_eval"], device, with_f1=False),
             "test": evaluate_classifier(model, loaders["test"], device, with_f1=True),
             "val": evaluate_classifier(model, loaders["val"], device, with_f1=False),
             "profile": profile(model, f"resnet_{tag}"),
             "serialized_fp32_mb": state_dict_bytes(model.state_dict()) / 2**20,
             "unique_parameters": unique_parameter_count(model)}
        return r

    log("[Task 3] FP32 baseline")
    res["models"]["fp32"] = metrics(fp_model, "fp32", "FP32 kernels")
    for bits in (8, 4):
        tag = f"qat_w{bits}a8"
        log(f"[Task 3] QAT W{bits}A8")
        set_seed(qcfg.seed)
        loaders["train"].generator.manual_seed(qcfg.seed)
        q = prepare_qat(fp_model, bits, 8).to(device)
        q, hist = train_qat(q, loaders, device, qcfg.epochs, qcfg.lr, freeze_after_epoch=qcfg.freeze_after_epoch,
                            init_batches=qcfg.init_batches, seed=qcfg.seed, log=log)
        res["histories"][tag] = hist
        fp_model.cpu()
        q.zero_grad(set_to_none=True)
        import gc
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        r = metrics(q, tag, "simulated quantization (fake-quant STE layers, FP32 conv kernels)")
        export, meta = export_qat_state_dict(q)
        r["storage"] = {"integer_export_mb": state_dict_bytes({"state_dict": export, "metadata": meta}) / 2**20,
                        "theoretical_packed_mb": theoretical_packed_bytes_qat(q)["total_bytes"] / 2**20,
                        "int4_packed": bits <= 4,
                        "simulated_runtime_fp32_mb": state_dict_bytes({k: v for k, v in q.state_dict().items()}) / 2**20}
        errs = qat_layer_errors(q, loaders["val"], device)
        r["layer_errors"] = errs
        r["error_summary"] = {"E_W": float(np.mean([e["E_W"] for e in errs.values()])),
                              "E_X": float(np.mean([e["E_X"] for e in errs.values()])),
                              "clip_frac": float(np.mean([e["clip_frac"] for e in errs.values()]))}
        r["train_seconds"] = hist["total_train_seconds"]
        res["models"][tag] = r
        torch.save(q.state_dict(), qcfg.root / f"resnet18_{tag}.pt")
        del q
    res["figures"] = [plot_training_curves({"W8A8": res["histories"]["qat_w8a8"], "W4A8": res["histories"]["qat_w4a8"]},
                                           qcfg.root / "figures" / "qat_training_curves.png")]
    save_json(res, out_path)
    return res

In [ ]:
"""Builds REPORT.md from the JSON results written by the experiment runners.

The explanatory text is fixed. Every number, table and figure reference is filled from the measured results, and
the data-driven sentences (best method per precision, static vs dynamic deltas, SmoothQuant and GPTQ effects) are
computed from the same JSON so that the report never disagrees with the notebook outputs.
"""


import json
from pathlib import Path

In [ ]:
def _f(x, nd=3, unit=""):
    if x is None:
        return "n/a"
    if isinstance(x, bool):
        return "yes" if x else "no"
    if isinstance(x, (int,)) and not isinstance(x, bool):
        return f"{x:,}{unit}"
    try:
        xf = float(x)
    except Exception:
        return str(x)
    if abs(xf) >= 1e5:
        return f"{xf:,.0f}{unit}"
    if abs(xf) < 1e-3 and xf != 0:
        return f"{xf:.2e}{unit}"
    return f"{xf:.{nd}f}{unit}"

In [ ]:
def table(headers: list[str], rows: list[list], align_first_left=True) -> str:
    out = ["| " + " | ".join(headers) + " |", "|" + "|".join(":---" if (i == 0 and align_first_left) else "---:" for i in range(len(headers))) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(c) for c in r) + " |")
    return "\n".join(out)

In [ ]:
def _rel(path: str, root: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(root.resolve()))
    except Exception:
        return path

In [ ]:
def _profile_rows(prof: dict) -> list[list]:
    return [
        ["Latency (ms, mean ± std over measured passes)", f"{_f(prof['latency_ms_mean'])} ± {_f(prof['latency_ms_std'])}"],
        ["Latency median / min (ms)", f"{_f(prof['latency_ms_median'])} / {_f(prof['latency_ms_min'])}"],
        ["Sequences per second", _f(prof["items_per_s"])],
        ["Input tokens per second", _f(prof["tokens_per_s"], 1)],
        ["Peak GPU allocated memory (MiB)", _f(prof.get("peak_gpu_allocated_mb"), 1)],
        ["Peak process CPU memory, sampled RSS (MiB)", _f(prof.get("peak_process_rss_mb_sampled"), 1)],
        ["Average process CPU utilization (%)", _f(prof.get("avg_process_cpu_percent"), 1)],
        ["Average GPU utilization (%)", _f(prof.get("avg_gpu_util_percent"), 1)],
        ["Average GPU power (W)", _f(prof.get("avg_gpu_power_w"), 2)],
        ["GPU energy per forward pass (mJ)", _f(prof.get("gpu_energy_per_pass_mj"), 2)],
        ["Average CPU power (W)", _f(prof.get("avg_cpu_power_w"), 2)],
        ["CPU energy per forward pass (mJ)", _f(prof.get("cpu_energy_per_pass_mj"), 2)],
        ["GPU energy per MAC / per FLOP (pJ)", f"{_f(prof.get('gpu_energy_per_mac_pj'), 2)} / {_f(prof.get('gpu_energy_per_flop_pj'), 2)}"],
        ["CPU energy per MAC / per FLOP (pJ)", f"{_f(prof.get('cpu_energy_per_mac_pj'), 2)} / {_f(prof.get('cpu_energy_per_flop_pj'), 2)}"],
        ["Power sampling", f"{prof.get('power_samples')} samples at {prof.get('sample_interval_s')} s over {prof.get('power_loop_passes')} passes ({_f(prof.get('power_loop_seconds'), 2)} s)"],
        ["torch.profiler FLOPs per pass (matmul/addmm/conv only)", _f(prof.get("profiler_flops_per_pass"), 0)],
    ]

In [ ]:
def _method_table(results: dict, label: str) -> str:
    rows = []
    for cname, r in results.items():
        p, lm, st = r["profile"], r["lm"], r["storage"]
        rows.append([f"{label} {cname}" + (f" (α={r['alpha']})" if r.get("alpha") is not None else ""), _f(lm["accuracy"], 2), _f(lm["nll"], 4), _f(lm["ppl"], 2),
                     _f(st["integer_export_mb"], 1), _f(st["theoretical_packed_mb"], 1), f"{_f(p['latency_ms_mean'], 2)} ± {_f(p['latency_ms_std'], 2)}",
                     _f(p["tokens_per_s"], 0), _f(p.get("peak_gpu_allocated_mb"), 0), _f(p.get("peak_process_rss_mb_sampled"), 0),
                     _f(p.get("avg_gpu_power_w"), 1), _f(p.get("gpu_energy_per_pass_mj"), 1), _f(r.get("calibration_seconds"), 1)])
    return table(["Configuration", "Acc %", "NLL", "PPL", "Int export MiB", "Packed MiB (theory)", "Latency ms", "tok/s", "GPU MiB", "RSS MiB", "GPU W", "mJ/pass", "Calib s"], rows)

In [ ]:
def _error_table(results: dict, label: str) -> str:
    rows = []
    for cname, r in results.items():
        e = r["error_summary"]
        rows.append([f"{label} {cname}", _f(e["E_W"], 5), _f(e["E_X"], 5), _f(e["E_Y_w"], 5), _f(e["E_Y"], 5), _f(e["clip_frac"], 6)])
    return table(["Configuration", "mean E_W", "Mean E_X", "mean E_Y (weights only)", "mean E_Y (W + A8)", "static clip fraction"], rows)

In [ ]:
def _per_type_table(results: dict, label: str) -> str:
    rows = []
    for cname, r in results.items():
        for ptype, e in r["error_summary"]["per_type"].items():
            rows.append([f"{label} {cname}", ptype, _f(e["E_W"], 5), _f(e["E_X"], 5), _f(e["E_Y_w"], 5), _f(e["E_Y"], 5)])
    return table(["Configuration", "Projection", "E_W", "E_X", "E_Y (W only)", "E_Y (W + A8)"], rows)

In [ ]:
def _stats_md(rows: list[dict]) -> str:
    return table(["Projection", "max abs X", "p99 abs X", "X max/median channel", "top channels", "max abs W", "p99 abs W", "W max/median channel"],
                 [[r["projection"], _f(r["act_max"], 2), _f(r["act_p99"], 3), _f(r["act_max/median_ch"], 1), r["act_top_channels"],
                   _f(r["w_max"], 2), _f(r["w_p99"], 3), _f(r["w_max/median_ch"], 1)] for r in rows])

In [ ]:
def _layerwise_table(results: dict, label: str, key: str = "layer_errors") -> str:
    """Full 48-row layer-wise table for one method (all four configs side by side, E_Y with W + A8)."""
    names = list(next(iter(results.values()))[key].keys())
    cfgs = list(results.keys())
    rows = []
    for n in names:
        rows.append([n.replace("transformer.h.", "h")] + [_f(results[c][key][n]["E_W"], 5) for c in cfgs] + [_f(results[c][key][n]["E_Y"], 5) for c in cfgs])
    return table(["Layer"] + [f"E_W {c}" for c in cfgs] + [f"E_Y {c}" for c in cfgs], rows)

In [ ]:
def build_report(results_dir: str | Path, out_path: str | Path | None = None, author: str = "Ahmed Jawed (STUDENT_ID_REMOVED)") -> str:
    root = Path(results_dir)
    t1 = load_json(root / "task1_baseline.json")
    ptq = load_json(root / "task2_1_linear_ptq.json")
    sq = load_json(root / "task2_2_smoothquant.json")
    gq = load_json(root / "task2_3_gptq.json")
    t3p = root / "task3_qat.json"
    t3 = load_json(t3p) if t3p.exists() else None
    env, data, macs = t1["environment"], t1["data"], t1["macs"]
    md: list[str] = []
    A = md.append

    # ------------------------------------------------------------------ header
    A(f"# AI624 Assignment 3: Quantization of GPT-2 and ResNet-18\n\n{author}\n")
    A('This report covers GPT-2 PTQ, SmoothQuant, GPTQ and ResNet-18 QAT. The implementations are included in the main notebook and use the assignment equations and original papers. Tables are generated from the saved experiment results.\n')
    A("## Environment and rerunning\n")
    A(table(["Item", "Value"], [[k, _f(v)] for k, v in env.items()]))
    A(f"\nSeeds: global seed {t1['data']['calibration']['seed']} for calibration sampling, `torch.manual_seed({t1['data']['calibration']['seed']})` for all runs. "
      f"Reproduction: execute the self-contained notebook top to bottom with `SMOKE=False` and `reuse=False` (see README).\n")

    A('Histograms and 99th percentiles use deterministic samples of up to 200,000 values. Channel maxima and local errors use the first 16 validation blocks. Distribution overlays use the first four of these blocks. Final metrics use the full test set.\n\nDynamic A8 has no fixed calibration range, so its clipping column is n/a. Each input tensor defines its own range. For ResNet profiling, one sequence means one image, so throughput is images per second.\n')

    # ------------------------------------------------------------------ Task 1
    A("# Task 1: GPT-2 baseline and profiling\n")
    A("## 1.1 Architecture and quantization scope\n")
    arch = t1["architecture"]
    A("GPT-2 small is a decoder-only Transformer with learned token embeddings (`wte`, 50257 × 768), learned absolute position "
      "embeddings (`wpe`, 1024 × 768), 12 identical pre-LayerNorm blocks, a final LayerNorm `ln_f` and a vocabulary projection "
      f"tied to `wte` (tied: {arch['lm_head_tied_to_wte']}). Hidden size 768, 12 heads of 64 dimensions, MLP width 3072.\n")
    A("Order of operations inside one block for hidden states $h \\in \\mathbb{R}^{L \\times 768}$:\n")
    A("1. $x = \\mathrm{LN}_1(h)$ (pre-LayerNorm: normalise over the 768 features, then affine $\\gamma, \\beta$).\n"
      "2. $[Q, K, V] = x\\,W_{qkv} + b_{qkv}$ with `attn.c_attn` (768 → 2304), split into 12 heads of 64.\n"
      "3. Causal multi-head self-attention: $A_h = \\mathrm{softmax}\\big(Q_h K_h^{\\top}/\\sqrt{64} + M\\big)$ with the causal mask $M$, "
      "context $C_h = A_h V_h$. Heads are concatenated to 768 features.\n"
      "4. Attention output projection `attn.c_proj` (768 → 768) and residual: $h' = h + C\\,W_o + b_o$.\n"
      "5. $y = \\mathrm{LN}_2(h')$. MLP input projection `mlp.c_fc` (768 → 3072). GELU. MLP output projection `mlp.c_proj` (3072 → 768). Residual $h'' = h' + \\mathrm{GELU}(y W_1 + b_1) W_2 + b_2$.\n")
    A("Quantization scope (Task 2): the weights and input activations of the four projections per block are quantized. Biases, "
      "embeddings, the tied vocabulary head, LayerNorm, GELU, softmax, residual additions and the two attention matrix "
      "multiplications stay in FP32, and no KV cache is used (`use_cache=False`, full 128-token forward passes).\n")
    A(table(["Projection", "Module", "Type", "Weight shape [in, out]"],
            [[p, f"transformer.h[i].{m}", arch["projection_types"][m], arch["projection_weight_shapes"][m]]
             for p, m in (("Q, K, V", "attn.c_attn"), ("Attention output", "attn.c_proj"), ("MLP input", "mlp.c_fc"), ("MLP output", "mlp.c_proj"))]))
    A(f"\n{arch['weight_layout']} Per-output-channel weight scales are therefore taken over the *columns* of the Conv1D weight "
      "(`amax(dim=0)`), and the dimension check `in_features = weight.shape[0]`, `out_features = weight.shape[1]` is asserted "
      "when the module is replaced.\n")

    A("## 1.2 Baseline language-model performance\n")
    A("Data protocol: WikiText-2 (`wikitext-2-raw-v1`), GPT-2 tokenizer, nonempty examples concatenated with `<|endoftext|>` "
      "as separator, nonoverlapping 128-token blocks, incomplete tail discarded, 127 targets scored per block (the first token "
      "has no context). Loss is summed over all scored targets and divided by their count (not an average of batch perplexities).\n")
    A(table(["Split", "Nonempty examples", "Tokens incl. separators", "Blocks of 128", "Scored targets", "Discarded tail tokens"],
            [[s, _f(d["nonempty_examples"]), _f(d["total_tokens_incl_separators"]), _f(d["blocks"]), _f(d["scored_targets"]), d["discarded_tail_tokens"]]
             for s, d in data.items() if s in ("train", "validation", "test")]))
    cal = data["calibration"]
    A(f"\nCalibration: {cal['n']} training blocks sampled with seed {cal['seed']} (first indices {cal['indices_head']}), reused "
      f"unchanged by every PTQ method. Evaluation/calibration batch size {data['eval_batch_size']}, profiling batch size 1. "
      f"Test blocks used: {data['test_blocks_used']} of {data['test']['blocks']}. Validation blocks used for α selection: {data['val_blocks_used']}.\n")
    A("$$\\mathrm{Accuracy} = \\frac{100}{N}\\sum_{t \\in T} \\mathbf{1}\\big[\\arg\\max_v p(v\\mid x_{<t}) = x_t\\big],\\qquad "
      "\\mathrm{NLL} = -\\frac{1}{N}\\sum_{t\\in T}\\log p(x_t\\mid x_{<t}),\\qquad \\mathrm{PPL} = e^{\\mathrm{NLL}}$$\n")
    lm = t1["lm"]
    A(table(["Metric", "FP32 baseline"], [["Top-1 next-token accuracy (%)", _f(lm["accuracy"], 2)], ["Mean NLL (nats)", _f(lm["nll"], 4)],
                                           ["Perplexity", _f(lm["ppl"], 3)], ["Scored targets", _f(lm["scored_targets"])],
                                           ["Unique parameters (tied counted once)", _f(t1["unique_parameters"])],
                                           ["Serialized FP32 size (MiB, torch.save of state_dict)", _f(t1["serialized_fp32_mb"], 2)]]))

    A("\n## 1.3 Baseline profiling\n")
    A(f"""I profile one fixed 128-token test block at batch size 1 with `use_cache=False`, evaluation mode and gradients disabled. There are {t1['profile']['warmup_passes']} warm-up passes and {t1['profile']['measured_passes']} timed passes. Each timed pass is synchronized before and after execution. Loading and tokenization are excluded.

Latency is measured with `time.perf_counter`. The PyTorch profiler runs separately for the operator breakdown and FLOP estimate. Power and utilization are sampled during another repeated-pass loop lasting {t1['profile'].get('power_loop_seconds', 0):.1f} seconds. Energy per pass is the trapezoidal integral of sampled power divided by the number of passes.
""")
    A("$$E_{\\mathrm{forward}} = \\frac{1}{K}\\int_0^T P(t)\\,dt,\\qquad E_{\\mathrm{MAC}} = \\frac{E_{\\mathrm{forward}}}{\\mathrm{MACs}_{\\mathrm{forward}}},\\qquad "
      "E_{\\mathrm{FLOP}} = \\frac{E_{\\mathrm{forward}}}{\\mathrm{FLOPs}_{\\mathrm{forward}}}$$\n")
    mb = macs["macs_breakdown"]
    A("MAC counting convention: 1 MAC = 2 FLOPs (one multiply and one add). Matrix products only. The elementwise operators "
      "(LayerNorm, GELU, softmax, residual and bias additions, embedding lookups) are estimated separately and are not MACs. "
      "The profiler's FLOP count covers `addmm`/`mm` (projections and vocabulary head) but not the fused scaled-dot-product "
      "attention kernel, which explains the small gap to the analytic total.\n")
    A(table(["Component (per forward pass, 128 tokens)", "MACs", "FLOPs"],
            [["Attention projections (c_attn + c_proj, 12 blocks)", _f(mb["attn_projections"]), _f(2 * mb["attn_projections"])],
             ["Attention matrix multiplications (QKᵀ and PV, 12 blocks)", _f(mb["attn_matmuls"]), _f(2 * mb["attn_matmuls"])],
             ["MLP projections (c_fc + c_proj, 12 blocks)", _f(mb["mlp_projections"]), _f(2 * mb["mlp_projections"])],
             ["Vocabulary head (tied, 128 × 768 × 50257)", _f(mb["vocab_head"]), _f(2 * mb["vocab_head"])],
             ["Total", f"{_f(macs['macs_total'])}", f"{_f(macs['flops_total'])}"],
             ["Elementwise operations (estimate, FLOPs, not MACs)", "", _f(macs["other_elementwise_flops_estimate"])]]))
    A("\n" + table(["Metric", "FP32 baseline"], _profile_rows(t1["profile"])))
    notes = t1["profile"].get("notes", {})
    if notes:
        A("\nUnavailable or limited sensors:\n")
        for k, v in notes.items():
            A(f"- {k}: {v}")
    A("")

    # ------------------------------------------------------------------ Task 2
    A("# Task 2: Post-training quantization of GPT-2\n")
    A("## Quantizer settings\n")
    A("$$q = \\mathrm{clip}\\big(\\mathrm{round}(r/S) + Z,\\ q_{\\min},\\ q_{\\max}\\big),\\qquad \\hat r = S\\,(q - Z)$$\n")
    A("Weights: symmetric, per output channel $c$ (column of the Conv1D weight), $Q_b = 2^{b-1}-1$ (127 for W8, 7 for W4):\n")
    A("$$S_W^{(c)} = \\frac{\\max_i |W_{ic}|}{Q_b},\\qquad Z_W = 0,\\qquad q \\in [-Q_b, Q_b]$$\n")
    A("Activations: asymmetric, per tensor, unsigned 8 bit, range widened to include zero:\n")
    A("$$S_X = \\frac{x_{\\max} - x_{\\min}}{255},\\qquad Z_X = \\mathrm{clip}\\big(\\mathrm{round}(-x_{\\min}/S_X),\\,0,\\,255\\big)$$\n")
    A("Zero-range tensors: when $x_{\\max} = x_{\\min}$ (or a weight column is all zeros) the scale is replaced by 1 so that "
      "$q = Z$ and $\\hat r = 0$ exactly, avoiding division by zero. Static A8 freezes one $(S_X, Z_X)$ per projection from the "
      "aggregated min/max over all calibration inputs. Dynamic A8 recomputes $(S_X, Z_X)$ from the current input tensor at "
      "every forward pass. Weight scales are always fixed before inference.\n")
    A('All 12 configurations use simulated quantization. Integer weights are dequantized when the model is built. Activations are quantized and dequantized during each forward pass, and projections use FP32 kernels. Dynamic A8 also calculates the current min/max range.\n\nThe notebook checks the integer projection formula against this simulated path. The profiled models do not use integer kernels. I report actual integer export size, theoretical packed size and complete simulated runtime state separately. The exports include quantization parameters, biases and retained FP32 tensors.\n')

    # 2.1
    A("## 2.1 Static and dynamic linear PTQ\n")
    A("### Quantized projection expression\n")
    A("With $X = S_X(q_X - Z_X)$ and $W_{:c} = S_W^{(c)} q_{W,:c}$, the projection $Y = XW + b$ becomes, for output channel $c$:\n")
    A("$$Y_{tc} = S_X S_W^{(c)} \\sum_k (q_{X,tk} - Z_X)\\, q_{W,kc} + b_c "
      "= S_X S_W^{(c)}\\Big[\\underbrace{\\sum_k q_{X,tk}\\, q_{W,kc}}_{\\text{int32 accumulation}} - Z_X \\underbrace{\\sum_k q_{W,kc}}_{\\text{per-column constant}}\\Big] + b_c$$\n")
    A('The first sum multiplies uint8 activations and int8 weights with int32 accumulation. Its magnitude is bounded by $3072 \\cdot 255 \\cdot 127 < 2^{31}$. The weight sum is precomputed for the zero-point correction. The result is rescaled by $S_X S_W^{(c)}$, then the FP32 bias is added. The notebook function `integer_projection_reference` checks this expression using int64 accumulation. The difference from simulated execution is reported in the notebook.\n')
    A("### Results on the test split\n")
    A(_method_table(ptq, "Linear PTQ"))
    A("\nStorage includes the per-channel weight scales (one FP32 per output channel), the static activation scale and zero "
      "point per projection, FP32 biases and all unquantized tensors. 4-bit weights are packed two per byte in the export.\n")
    A("### Layer-wise reconstruction errors (validation inputs, " + str(data["analysis_blocks"]) + " fixed blocks)\n")
    A("$$E_A = \\frac{\\|A - \\hat A\\|_F^2}{\\|A\\|_F^2 + \\epsilon}$$\n")
    A("E_W compares each FP32 weight with its dequantized version. E_X compares each projection input with its "
      "quantized-dequantized version on the FP32 model's inputs. E_Y compares the projection output of the FP32 model with the "
      "output produced from the same input by the quantized projection (weights only, and weights plus A8). Means over the 48 projections:\n")
    A(_error_table(ptq, "Linear PTQ"))
    A("\nPer projection type:\n")
    A(_per_type_table(ptq, "Linear PTQ"))
    A("\n" + _static_dynamic_text(ptq))

    # 2.2
    A("## 2.2 SmoothQuant and outlier analysis\n")
    A("### Distributions before smoothing (blocks 0, 5, 11, validation inputs)\n")
    A(_stats_md(sq["stats_original"]))
    A("\n`max/median channel` is the ratio between the largest per-input-channel maximum and the median channel maximum. "
      "Values far above 1 mean a few channels dominate the per-tensor range. `top channels` lists the three input channels with "
      "the largest magnitude. Per-channel plots and histograms: `results/figures/dist_smoothquant_*`.\n")
    A("### Scaling\n")
    A("$$X' = X D^{-1},\\quad W' = D W,\\quad D = \\mathrm{diag}(s),\\quad s_j = \\frac{(\\max_t |X_{tj}| + \\epsilon)^{\\alpha}}{(\\max_c |W_{jc}| + \\epsilon)^{1-\\alpha}}$$\n")
    A('Activation maxima come only from calibration data. For `attn.c_attn` and `mlp.c_fc`, I fold $D^{-1}$ into the LayerNorm parameters ($\\gamma/s$, $\\beta/s$) and apply $DW$ to the projection weights.\n\nFor `attn.c_proj`, the attention context is linear in V, so its input scaling is folded into the V slice of `c_attn` weights and bias. The input to `mlp.c_proj` follows GELU, so it is scaled explicitly at runtime. This elementwise work is included in profiling. The residual stream is unchanged, and shared branches remain consistent.\n')
    A("### Equivalence before quantization\n")
    A(table(["α", "max abs(delta logits)", "mean abs(delta logits)", "relative output norm error", "max abs(logits)"],
            [[a, _f(e["max_abs_diff"], 6), _f(e["mean_abs_diff"], 6), _f(e["relative_output_error"], 8), _f(e["logits_abs_max"], 1)] for a, e in sq["equivalence"].items()]))
    A("\n### Alpha selection using validation perplexity\n")
    rows = []
    for cname, srch in sq["alpha_search"].items():
        rows.append([cname] + [_f(srch[str(a)]["ppl"], 3) for a in sq["alphas"]] + [sq["selected_alpha"][cname]])
    A(table(["Configuration"] + [f"val PPL α={a}" for a in sq["alphas"]] + ["selected α"], rows))
    A("\n### Results on the test split\n")
    A(_method_table(sq["configs"], "SmoothQuant"))
    A("\n### Distributions after smoothing (same validation sequences)\n")
    for k in sorted(k for k in sq if k.startswith("stats_smoothed_alpha_")):
        A(f"α = {k.split('_')[-1]}:\n")
        A(_stats_md(sq[k]))
        A("")
    A("### Reconstruction errors and clipping\n")
    A(_error_table(sq["configs"], "SmoothQuant"))
    A("\n" + _per_type_table(sq["configs"], "SmoothQuant"))
    A("\n### Analysis\n")
    A(_smoothquant_text(ptq, sq))

    # 2.3
    A("## 2.3 GPTQ and reconstruction analysis\n")
    A("### Distributions before quantization\n")
    A('GPTQ starts from the original model without smoothing. Its starting distributions are the same ones shown in Section 2.2, using the same validation blocks and statistics.\n')
    A("### Method\n")
    A("$$\\min_{\\hat W}\\ \\|XW - X\\hat W\\|_F^2,\\qquad H = 2X^{\\top}X + \\lambda I$$\n")
    A("For a layer with calibration inputs $X \\in \\mathbb{R}^{N \\times d_{in}}$ every output channel is an independent least "
      "squares problem sharing the Hessian $H$. Quantizing weight $i$ of a row $w$ to $q_i$ and re-optimising the remaining "
      "free weights (OBS) gives the optimal compensation\n")
    A("$$\\delta w = -\\frac{w_i - q_i}{[H^{-1}]_{ii}}\\,[H^{-1}]_{:,i},\\qquad \\Delta\\mathcal{L} = \\frac{(w_i - q_i)^2}{2\\,[H^{-1}]_{ii}}$$\n")
    A('The inverse Hessian determines how a rounding error is shared among the remaining weights. GPTQ uses the upper Cholesky factor U of $H^{-1}$ in a fixed column order. Row i, restricted to columns after i, gives the update direction after earlier coordinates have been removed. This implements the sequence of inverse-Hessian updates in a more stable form.\n')
    q8, q4 = gq["quantization"]["W8"], gq["quantization"]["W4"]
    A(table(["Setting", "Value"], [
        ["Hessian", "H = (2/N) Σ x xᵀ accumulated in FP32 over all calibration tokens of the layer (N = " + _f(q8["layers"][0].get("tokens", q8["n_calib_sequences"] * 128)) + ")"],
        ["Damping rule", f"λ = {q8['percdamp']} × mean(diag H) added to the diagonal (dead inputs with H_ii = 0 get H_ii = 1 and w_i = 0)"],
        ["Numerical stabilisation", "Cholesky of H, `cholesky_inverse`, upper Cholesky of H⁻¹. On failure λ is multiplied by 10 and retried (retries used: "
                                    + str(sum(l["cholesky_retries"] for l in q8["layers"] + q4["layers"])) + ")"],
        ["Block size", f"{q8['block_size']} columns (lazy batch update of the remaining columns after each block)"],
        ["Quantization order", q8["ordering"]],
        ["Weight grid", "same symmetric per-output-channel scales and integer range as linear PTQ, computed from the original FP32 weights"],
        ["Layer order", "block 0 → 11. Inside a block c_attn → attn.c_proj → c_fc → mlp.c_proj, each layer's inputs collected with the preceding layers already quantized"],
        ["Activations during GPTQ", "FP32 (A8 is applied afterwards. Static ranges are calibrated on the GPTQ-weight model)"],
        ["Quantization time W8 / W4 (s)", f"{_f(q8['total_seconds'], 1)} / {_f(q4['total_seconds'], 1)}"],
        ["Peak memory during GPTQ W8 / W4", f"GPU alloc {_f(q8.get('peak_gpu_allocated_mb'), 0)} / {_f(q4.get('peak_gpu_allocated_mb'), 0)} MiB. Process RSS {_f(q8.get('peak_process_rss_mb'), 0)} / {_f(q4.get('peak_process_rss_mb'), 0)} MiB"],
        ["Work device for the Hessian math", q8["work_device"]],
    ]))
    A("\n### Results on the test split\n")
    A(_method_table(gq["configs"], "GPTQ"))
    A("\n### Reconstruction errors vs round-to-nearest (identical validation inputs)\n")
    A(_error_table(gq["configs"], "GPTQ"))
    A("\n" + _per_type_table(gq["configs"], "GPTQ"))
    A("\n### Analysis\n")
    A(_gptq_text(ptq, sq, gq))

    # 2.4
    A("## 2.4 Consolidated results\n")


    rows = consolidated_rows(t1, ptq, sq, gq)
    A(table(["Configuration", "Acc %", "NLL", "PPL", "Size MiB (int export / FP32 baseline)", "Packed MiB (theory)", "GMACs", "GFLOPs", "Latency ms", "seq/s", "tok/s", "GPU MiB", "RSS MiB", "CPU %", "GPU %", "GPU W", "mJ/pass", "pJ/MAC", "mean E_Y", "α", "Execution"],
            [[r["configuration"], _f(r["accuracy_%"], 2), _f(r["nll"], 4), _f(r["ppl"], 2), _f(r["size_mb"], 1), _f(r["size_theoretical_packed_mb"], 1), _f(r["macs_G"], 2), _f(r["flops_G"], 2),
              f"{_f(r['latency_ms'], 2)} ± {_f(r['latency_std_ms'], 2)}", _f(r["seq_per_s"], 1), _f(r["tokens_per_s"], 0), _f(r["peak_gpu_mb"], 0), _f(r["peak_rss_mb"], 0),
              _f(r["cpu_util_%"], 0), _f(r["gpu_util_%"], 0), _f(r["gpu_power_w"], 1), _f(r["gpu_energy_mj"], 1), _f(r["energy_per_mac_pj"], 2), _f(r["E_Y_mean"], 5), _f(r["alpha"], 2), r["execution"]] for r in rows]))
    A("\n" + _consolidated_text(rows, t1))
    A("\n### Execution and storage\n")
    A('The measured models use FP32 kernels, so projection MAC counts stay unchanged. Quantizer and smoothing operations add work outside the conventional 2*MAC FLOP count. A real integer implementation would perform the same MAC count using integer arithmetic.\n\nA8 values and W8 weights need one byte each. Packed W4 weights need half a byte each. Integer accumulation would use int32, while unquantized operations remain FP32. Unpacking W4 to int8 removes its runtime payload advantage over W8. The T4 supports INT4 Tensor Core operations, but this implementation has no mixed W4A8 integer backend. Integer speedups and energy savings are therefore not measured here.\n\nActual exports include metadata and count tied embeddings once. Theoretical packed sizes exclude container overhead. All MiB values use 2**20 bytes. CPU energy is unavailable where RAPL cannot be accessed.\n')
    all_results = [(f"{label} {c}", r) for label, group in (("RTN",ptq),("SmoothQuant",sq["configs"]),("GPTQ",gq["configs"])) for c,r in group.items()]
    A(table(["Configuration","Complete runtime state MiB","GPU pJ/FLOP","CPU mJ/pass"],
            [[n,_f(r["storage"]["simulated_runtime_fp32_mb"],2),_f(r["profile"].get("gpu_energy_per_flop_pj"),2),_f(r["profile"].get("cpu_energy_per_pass_mj"),2)] for n,r in all_results]))
    A("Layer-wise tables (48 projections per method) follow in the appendix. Distribution figures are in `results/figures/`.\n")

    # ------------------------------------------------------------------ Task 3
    if t3:
        A("# Task 3: Quantization-aware training of ResNet-18 on CIFAR-10\n")
        A("## 3.1 Model setup\n")
        sp, env3 = t3["split"], t3["environment"]
        A(f"""I use `{t3['checkpoint'].split('/')[-1]}` from `huggingface.co/edadaltocg/resnet18_cifar10`. It is loaded into a timm ResNet-18 with a CIFAR stem: a 3x3 stride-1 first convolution and no max-pool. Inputs are 32x32 and use the checkpoint's normalization. Training uses random crops with padding 4 and horizontal flips. Evaluation has no augmentation.
""")
        A(f"""I split the CIFAR-10 training set once using seed {sp['seed']} into {sp['train_size']:,} training and {sp['val_size']:,} validation images. The first validation indices are {sp['val_indices_head'][:5]}. The {sp['test_size']:,} test images are used only for final evaluation. Both QAT runs start from the same checkpoint and use the same split and budget.

The public checkpoint was pretrained on the full CIFAR-10 training set, so it has already seen the validation images. Its high validation accuracy is therefore optimistic and is a limitation of checkpoint selection here. Test images were held out and were not used for selection.
""")
        A(table(["Item", "Value"], [[k, _f(v)] for k, v in env3.items()]))
        A("\n## 3.2 QAT implementation\n")
        c = t3["histories"]["qat_w8a8"]["config"]
        A('I apply fake quantization to the weights and inputs of all 20 convolutions and the final linear layer. Weight scales are symmetric per output channel and are recomputed from FP32 master weights at each step. Activations use asymmetric per-tensor unsigned A8. Biases remain FP32.\n\nThe forward pass clips $x$ to $x_c$, quantizes and dequantizes it to $\\hat x$, then returns $x_c + \\mathrm{sg}(\\hat x-x_c)$, where sg stops the gradient. The STE passes gradients unchanged inside the clipping range. Outside it, the clip has zero derivative. The weight clipping range is $[-Q_b S_W^{(c)}, Q_b S_W^{(c)}]$.\n')
        A(f"""Activation observers start with exact min/max values from {c['init_batches']} training batches while fake quantization is disabled. During training, they use an exponential moving average with momentum 0.9. They are frozen after epoch {c['freeze_after_epoch']} and during all validation and test evaluations.

BatchNorm stays separate and in FP32. It uses batch statistics during training and running statistics during evaluation. I do not fold it into the convolutions because folding would change the weights and require quantization again. Final evaluation uses fake quantization with frozen observers.
""")
        A(table(["Hyperparameter", "Value"], [["Optimizer", f"{c['optimizer']} momentum {c['momentum']} nesterov, weight decay {c['weight_decay']}"],
                                               ["Learning rate", f"{c['lr']} with cosine annealing over {c['epochs']} epochs"],
                                               ["Batch size", t3["config"]["batch_size"]], ["Epochs", c["epochs"]],
                                               ["Augmentation", "random crop (pad 4) + horizontal flip"],
                                               ["Observer", f"min/max init over {c['init_batches']} batches, EMA momentum 0.9, frozen after epoch {c['freeze_after_epoch']}"],
                                               ["Seed", c["seed"]], ["Checkpoint selection", "best validation top-1 (test set never used)"],
                                               ["Training time W8A8 / W4A8 (s)", f"{_f(t3['models']['qat_w8a8']['train_seconds'], 0)} / {_f(t3['models']['qat_w4a8']['train_seconds'], 0)}"],
                                               ["Best epoch W8A8 / W4A8", f"{t3['histories']['qat_w8a8']['best_epoch']} / {t3['histories']['qat_w4a8']['best_epoch']}"]]))
        A("\n## 3.3 Results and profiling\n")
        m3 = t3["models"]
        rows = []
        for tag, label in (("fp32", "FP32 baseline"), ("qat_w8a8", "QAT W8A8"), ("qat_w4a8", "QAT W4A8")):
            r = m3[tag]
            st = r.get("storage", {})
            rows.append([label, _f(r["train"]["top1"], 2), _f(r["test"]["top1"], 2), _f(r["test"]["top5"], 2), _f(r["test"]["macro_f1"], 2),
                         _f(st.get("integer_export_mb", r["serialized_fp32_mb"]), 2), _f(st.get("theoretical_packed_mb", r["serialized_fp32_mb"]), 2),
                         f"{_f(r['profile']['latency_ms_mean'], 2)} ± {_f(r['profile']['latency_ms_std'], 2)}", _f(r["profile"]["items_per_s"], 1),
                         _f(r["profile"].get("peak_gpu_allocated_mb"), 0), _f(r["profile"].get("peak_process_rss_mb_sampled"), 0),
                         _f(r["profile"].get("avg_gpu_power_w"), 1), _f(r["profile"].get("gpu_energy_per_pass_mj"), 2), r["execution"]])
        A(table(["Model", "Train top-1", "Test top-1", "Test top-5", "Test macro-F1", "Size MiB (export)", "Packed MiB (theory)", "Latency ms", "img/s", "GPU MiB", "RSS MiB", "GPU W", "mJ/pass", "Execution"], rows))
        mb3 = t3["macs"]
        A(f"\nMACs per 1×3×32×32 forward pass: {_f(mb3['macs_total'])} (conv {_f(mb3['macs_breakdown']['conv'])}, fc {_f(mb3['macs_breakdown']['linear'])}), "
          f"FLOPs {_f(mb3['flops_total'])}. Identical for all three models (simulated quantization keeps the FP32 kernels). "
          "Profiling protocol as in Task 1 (same warm-up/measured passes, batch size 1).\n")
        A(table(["Metric"] + ["FP32", "QAT W8A8", "QAT W4A8"], [[row[0]] + [ _profile_rows(m3[t]["profile"])[i][1] for t in ("fp32", "qat_w8a8", "qat_w4a8")] for i, row in enumerate(_profile_rows(m3["fp32"]["profile"]))]))
        A("\nLayer-wise normalized reconstruction errors of the fake-quantized weights (vs the FP32 master weights) and input "
          "activations (vs their FP32 values) on validation batches:\n")
        rows = []
        le8, le4 = m3["qat_w8a8"]["layer_errors"], m3["qat_w4a8"]["layer_errors"]
        for n in le8:
            rows.append([n, _f(le8[n]["E_W"], 6), _f(le8[n]["E_X"], 6), _f(le4[n]["E_W"], 6), _f(le4[n]["E_X"], 6)])
        A(table(["Layer", "E_W W8A8", "E_X W8A8", "E_W W4A8", "E_X W4A8"], rows))
        A(f"\nMeans: W8A8 E_W {_f(m3['qat_w8a8']['error_summary']['E_W'], 6)}, E_X {_f(m3['qat_w8a8']['error_summary']['E_X'], 6)}. "
          f"W4A8 E_W {_f(m3['qat_w4a8']['error_summary']['E_W'], 6)}, E_X {_f(m3['qat_w4a8']['error_summary']['E_X'], 6)}.\n")
        A(f"Training curves: `{_rel(t3['figures'][0], root.parent)}`\n\n![QAT curves]({_rel(t3['figures'][0], root.parent)})\n")
        A("## 3.4 Analysis\n")
        A(_qat_text(t3))

    # ------------------------------------------------------------------ appendix
    A("# Appendix: layer-wise results\n")
    for label, res in (("Linear PTQ", ptq), ("SmoothQuant", sq["configs"]), ("GPTQ", gq["configs"])):
        A(f"## {label}: E_W and E_Y (weights + A8) per projection\n")
        A(_layerwise_table(res, label))
        A("\n### Input activation reconstruction error E_X\n")
        A(table(["Layer"] + list(res), [[n.replace("transformer.h.","h")] + [_f(res[c]["layer_errors"][n]["E_X"],6) for c in res] for n in next(iter(res.values()))["layer_errors"]]))
        A("")
    A("## Figures\n")
    figs = sorted(p for p in (root / "figures").glob("*.png")) if (root / "figures").exists() else []
    for p in figs:
        rel = _rel(str(p), root.parent)
        if "_h5_" in p.name or p.name == "consolidated.png":
            A(f"\n![{p.stem}]({rel})\n")
        else:
            A(f"- [{p.stem}]({rel})")
    A("\n# References\n")
    A("1. Radford et al., Language Models are Unsupervised Multitask Learners, 2019.\n2. Xiao et al., SmoothQuant, ICML 2023, arXiv:2211.10438.\n"
      "3. Frantar et al., GPTQ, ICLR 2023, arXiv:2210.17323.\n4. Hugging Face Transformers, GPT-2 implementation and `openai-community/gpt2` checkpoint.\n"
      "5. Merity et al., Pointer Sentinel Mixture Models, ICLR 2017 (WikiText-2).\n6. PyTorch profiler documentation.\n"
      "7. E. Dadalto, detectors repository and `edadaltocg/resnet18_cifar10` checkpoint.\n")
    text = "\n".join(md)
    if out_path:
        Path(out_path).write_text(text)
    return text

In [ ]:
def _static_dynamic_text(ptq: dict) -> str:
    out = []
    for bits in (8, 4):
        a, b = ptq[f"static_W{bits}A8"], ptq[f"dynamic_W{bits}A8"]
        out.append(f"W{bits}A8: static/dynamic PPL is {a['lm']['ppl']:.2f}/{b['lm']['ppl']:.2f}. "
                   f"Mean E_X is {a['error_summary']['E_X']:.3g}/{b['error_summary']['E_X']:.3g}, "
                   f"and E_Y is {a['error_summary']['E_Y']:.3g}/{b['error_summary']['E_Y']:.3g}.\n\n")
    out.append("Static A8 uses the range collected from training calibration inputs. This range may miss later values, "
               "as the clipping fractions show. Dynamic A8 chooses a range for each input tensor. This changes the "
               "quantization grid while leaving the original input distribution unchanged. Both are affected by large activation channels.\n\n"
               "Weight errors match between static and dynamic A8 because the weights use the same grid. Comparing "
               "E_Y_w with E_Y shows the added effect of A8, although weight and activation errors are not exactly additive. "
               "Dynamic range calculation adds runtime work. The grid change explains accuracy differences.")
    return "".join(out)


In [ ]:
def _smoothquant_text(ptq: dict, sq: dict) -> str:
    out = []
    for name, r in sq['configs'].items():
        b = ptq[name]
        out.append(f"{name}, alpha={r['alpha']}: PPL is {r['lm']['ppl']:.2f} versus RTN {b['lm']['ppl']:.2f} "
                   f"(change {r['lm']['ppl']-b['lm']['ppl']:+.2f}). E_X is {r['error_summary']['E_X']:.3g} versus "
                   f"{b['error_summary']['E_X']:.3g}, E_W is {r['error_summary']['E_W']:.3g} versus "
                   f"{b['error_summary']['E_W']:.3g}, and E_Y in original coordinates is {r['error_summary']['E_Y']:.3g} "
                   f"versus {b['error_summary']['E_Y']:.3g}.\n\n")
    original = sq['stats_original']
    dominant = max(original, key=lambda z:z['act_max/median_ch'])
    out.append(f"The largest activation max/median channel ratio is {dominant['act_max/median_ch']:.1f} "
               f"at {dominant['projection']}. Weight outliers also occur. The plots show how smoothing reduces some "
               "activation ranges while moving difficulty into the weights. A larger alpha can help activations but "
               "make weight quantization harder. W4 has only 15 symmetric levels compared with 255 for W8, so it is more sensitive to this trade-off.\n\n"
               "Smoothing is not always better. The validation search includes poor results at alpha=0.25, and the selected "
               "W4 configurations have worse test perplexity than RTN. I select alpha separately for each configuration "
               "and recalibrate static ranges after smoothing. E_W and E_X use transformed tensors. E_Y is mapped back "
               "to the original coordinates. The check without quantization confirms that smoothing itself adds almost no error.\n\n"
               "Local errors and final perplexity can rank methods differently because errors pass through later nonlinear layers. "
               "Three scalings are folded into existing parameters. The input to the MLP output projection is scaled explicitly, and its cost is included in timing. "
               "All matrix products still use FP32 kernels.")
    return "".join(out)


In [ ]:
def _gptq_text(ptq: dict, sq: dict, gq: dict) -> str:
    out = []
    for name, g in gq['configs'].items():
        b = ptq[name]
        s = sq['configs'][name]
        out.append(f"{name}: GPTQ/RTN/SmoothQuant PPL is {g['lm']['ppl']:.2f}/{b['lm']['ppl']:.2f}/{s['lm']['ppl']:.2f}. "
                   f"GPTQ/RTN E_W is {g['error_summary']['E_W']:.3g}/{b['error_summary']['E_W']:.3g}. "
                   f"Weight-only E_Y is {g['error_summary']['E_Y_w']:.3g}/{b['error_summary']['E_Y_w']:.3g}. "
                   f"With A8, E_Y is {g['error_summary']['E_Y']:.3g}/{b['error_summary']['E_Y']:.3g}.\n\n")
    out.append("GPTQ uses calibration correlations to compensate for rounding errors in the remaining weights. "
               "This can increase weight error while reducing output error, which is what the paired E_W and E_Y_w values measure. "
               "The weight grid stays the same as RTN. Later layers receive calibration inputs from already quantized earlier layers.\n\n"
               "A8 can still limit the final result because GPTQ optimizes weights only. Static activation ranges are recalibrated "
               "on the GPTQ-weight model. Quantization time and peak memory are additional preparation costs. Inference uses "
               "the same simulated FP32 path as RTN, so runtime differences do not show an integer-kernel advantage.")
    return "".join(out)


In [ ]:
def _consolidated_text(rows: list[dict], t1: dict) -> str:
    base = rows[0]
    out = []
    for bits in (8, 4):
        for mode in ('static', 'dynamic'):
            group = [r for r in rows[1:] if r['configuration'].endswith(f'{mode}_W{bits}A8')]
            best = min(group, key=lambda r:r['ppl'])
            out.append(f"Best {mode} W{bits}A8: {best['configuration']}, PPL {best['ppl']:.2f} "
                       f"versus FP32 {base['ppl']:.2f} (change {best['ppl']-base['ppl']:+.2f}).\n\n")
    ratios = [r['latency_ms']/base['latency_ms'] for r in rows[1:]]
    out.append(f"Quantized latency is {min(ratios):.2f}x to {max(ratios):.2f}x FP32. The extra work comes from "
               "quantize/dequantize operations, dynamic range calculation and explicit smoothing. These models still use FP32 kernels. "
               "Lower power alone does not mean lower energy per pass. The energy per MAC and FLOP includes the whole forward pass and its overhead.\n\n"
               "Integer exports reduce storage for the selected projections. Embeddings and other FP32 tensors limit total model compression. "
               "Projection MAC counts stay unchanged. GPU memory includes the deployed model and forward workspace. CPU RSS also includes "
               "datasets and imports. Small timing differences should be read with their standard deviations because this is one hardware run.")
    return "".join(out)


In [ ]:
def _qat_text(t3: dict) -> str:
    models = t3['models']
    base = models['fp32']
    out = []
    for tag in ('qat_w8a8', 'qat_w4a8'):
        r = models[tag]
        h = t3['histories'][tag]
        largest = max(r['layer_errors'], key=lambda n:r['layer_errors'][n]['E_W'])
        out.append(f"{tag}: test top-1 is {r['test']['top1']:.2f}% versus FP32 {base['test']['top1']:.2f}% "
                   f"(change {r['test']['top1']-base['test']['top1']:+.2f} percentage points). Macro-F1 is "
                   f"{r['test']['macro_f1']:.2f} versus {base['test']['macro_f1']:.2f}. "
                   f"Validation accuracy changes from {h['val_acc'][0]:.2f}% to {h['val_acc'][-1]:.2f}%, "
                   f"and epoch {h['best_epoch']} is selected. The largest E_W is {r['layer_errors'][largest]['E_W']:.3g} "
                   f"at {largest}. Mean E_W/E_X is {r['error_summary']['E_W']:.3g}/{r['error_summary']['E_X']:.3g}. "
                   f"Latency is {r['profile']['latency_ms_mean']:.2f} ± {r['profile']['latency_ms_std']:.2f} ms "
                   f"({r['profile']['latency_ms_mean']/base['profile']['latency_ms_mean']:.2f}x FP32). "
                   f"The integer export is {r['storage']['integer_export_mb']:.2f} MiB versus "
                   f"{base['serialized_fp32_mb']:.2f} MiB FP32. Training takes {r['train_seconds']:.0f} seconds.\n\n")
    out.append("The training curves show the effects of optimization, augmentation, observer updates and BatchNorm, especially early on. "
               "W4 has larger weight error with its 15-level grid. W8 has 255 weight levels, and both use 256 activation codes. "
               "Average weight error alone does not determine final accuracy. One seed is not enough to say that one precision is always better.\n\n"
               "The public checkpoint saw the validation images during pretraining, so validation accuracy is optimistic. Test images were held out "
               "and were not used for selection. BatchNorm stays FP32 and is not folded. Fake quantization adds work around FP32 convolutions. "
               "Cached dequantized weights also use GPU memory. Packed exports reduce file size, but inference still uses FP32 kernels.\n\n"
               "Both QAT exports are smaller than FP32, with W4 giving the smaller file. In this simulated implementation, both variants use more GPU memory "
               "and energy per image than FP32. Their lower board power does not offset the longer execution. The latency standard deviations show variability, "
               "so this run does not establish a general W4 speed advantage. I did not run ResNet PTQ or an FP32 fine-tuning control, so these results do not compare QAT against those alternatives.")
    return "".join(out)


In [ ]:
# Focused controls for implementation correctness, independent of downloaded checkpoints.
from transformers import GPT2Config, GPT2LMHeadModel
torch.set_num_threads(1)
set_seed(624)
_dev = torch.device('cpu')
_fp = GPT2LMHeadModel(GPT2Config(n_layer=2, n_head=2, n_embd=16, n_positions=128,
                                vocab_size=32, use_cache=False)).eval()
_blocks = torch.randint(0, 32, (4,128))
_maxima = collect_channel_maxima(_fp, _blocks, 2, _dev)
_scales = compute_smooth_scales(_fp, _maxima, .75)
_sm = apply_smoothing(_fp, _scales)
_equiv = verify_equivalence(_fp, _sm, _blocks, _dev, n_blocks=4)
_q = quantize_smoothed(_sm, _fp, _scales, QuantSpec(8,8,'none'))
for _m in quant_modules(_q).values():
    _m.weight.copy_(_m.analysis_weight_ref)
_errors = layer_errors(_fp, _q, _blocks, 2, _dev)
_max_ey = max(v['E_Y'] for v in _errors.values())
assert _equiv['relative_output_error'] < 1e-5 and _max_ey < 1e-8
_qw, _qi = gptq_quantize_gpt2(_fp, _blocks, 2, _dev, 4, block_size=8, verbose=False)
assert len(_qw) == 8  # tensor and tuple Transformer-block return formats are normalized
_int_diffs = {}
for _mode in ('static','dynamic'):
    _m = QuantProjection('control', torch.randn(16,20), torch.randn(20), 4, act_mode=_mode)
    _x = torch.randn(2,3,16)
    if _mode == 'static':
        _m.observer.update(_x)
        _m.freeze_static_range()
    _diff = float((_m(_x) - integer_projection_reference(_x, _m)).abs().max())
    assert _diff < 1e-4
    _int_diffs[_mode] = _diff
for _n in (1,2,9,100):
    _x = torch.randint(-7,8,(_n,))
    assert torch.equal(_x, unpack_int4(pack_int4(_x),_n,_x.shape))
_x = torch.tensor([-2.,-.5,0.,.5,2.], requires_grad=True)
fake_quant_activation(_x, torch.tensor(-1.), torch.tensor(1.)).sum().backward()
assert _x.grad.tolist() == [0.,1.,1.,1.,0.]
_thread = ResourceSampler(_dev, interval=.005)
_thread.start()
_thread.ready.wait()
_thread.stop()
implementation_checks = {'smoothquant_fp_equivalence':_equiv,
                         'smoothquant_no_quant_max_E_Y':_max_ey,
                         'integer_reference_max_abs_difference':_int_diffs,
                         'int4_roundtrip':True, 'STE_gradient':True,
                         'GPTQ_sequential_block_return':True, 'profiler_thread_join':True}
display(implementation_checks)
del _fp, _sm, _q, _qw, _m, _x, _maxima, _scales
gc.collect()


In [ ]:
SMOKE = False   # True: tiny subsets to check the pipeline. False: full experiments

if SMOKE:
    cfg = Config(results_dir='results_smoke', test_blocks=8, val_blocks=8, analysis_blocks=4, n_calib=8, eval_batch_size=4, warmup=2, iters=5, min_power_seconds=0.5)
    qcfg = QATConfig(results_dir='results_smoke', epochs=2, warmup=2, iters=5, min_power_seconds=0.5, train_subset=1024)
else:
    cfg = Config(results_dir='results', eval_batch_size=16, reuse=False)
    qcfg = QATConfig(results_dir='results', epochs=8, lr=0.01, batch_size=128, data_source='hf', num_workers=2, reuse=False)
set_seed(cfg.seed)
cfg
cfg.root.mkdir(exist_ok=True)
save_json(implementation_checks, cfg.root / 'implementation_checks.json')
print('Fresh full run:', not SMOKE, '| seed:', cfg.seed, '| reuse:', cfg.reuse)

In [ ]:
t0 = time.time()
ctx = Context(cfg)   # loads GPT-2, tokenizes WikiText-2 into 128-token blocks, samples the 128 calibration sequences
print(f'context ready in {time.time()-t0:.0f}s on {ctx.device}')

arch = architecture_summary(ctx.model)
display(pd.DataFrame([{k: v for k, v in arch.items() if k not in ('projection_types', 'projection_weight_shapes')}]).T)
display(pd.DataFrame({'type': arch['projection_types'], 'weight shape [in, out]': arch['projection_weight_shapes']}))
print(ctx.model.transformer.h[0])

In [ ]:
t1 = run_task1(ctx)
display(pd.DataFrame([t1['data'][s] for s in ('train', 'validation', 'test')]).set_index('split'))
print('calibration:', t1['data']['calibration'])
display(pd.DataFrame([t1['lm']]).T.rename(columns={0: 'FP32 baseline'}))
print(f"unique parameters: {t1['unique_parameters']:,}   serialized FP32 size: {t1['serialized_fp32_mb']:.2f} MB")

In [ ]:
mb = t1['macs']
display(pd.DataFrame({'MACs': mb['macs_breakdown']}).assign(FLOPs=lambda d: 2 * d['MACs']))
print(f"total MACs {mb['macs_total']:,}  FLOPs {mb['flops_total']:,}  elementwise FLOPs (estimate) {mb['other_elementwise_flops_estimate']:,}")
prof = {k: v for k, v in t1['profile'].items() if k not in ('profiler_table_head', 'notes')}
display(pd.DataFrame([prof]).T.rename(columns={0: 'FP32 baseline'}))
print('sensor notes:', json.dumps(t1['profile']['notes'], indent=1))
print(t1['profile']['profiler_table_head'])
display(pd.DataFrame([t1['environment']]).T.rename(columns={0: 'environment'}))

In [ ]:


# quantizer sanity checks on one real weight
W = ctx.model.transformer.h[0].mlp.c_fc.weight.data
for b in (8, 4):
    q, s, deq = quantize_weight(W, b, channel_dim=1)
    print(f'W{b}: int range [{int(q.min())}, {int(q.max())}], scales per output channel {tuple(s.shape)}, normalized error {float(((W-deq)**2).sum()/(W**2).sum()):.3e}')
print('zero-range activation ->', activation_qparams(torch.tensor(0.), torch.tensor(0.)))
# integer execution reference vs simulated path (CPU, int64 accumulation)
import copy
cpu_model = copy.deepcopy(ctx.model).cpu()
qm_cpu = quantize_gpt2(cpu_model, QuantSpec(8, 8, 'dynamic'))
m = quant_modules(qm_cpu)['transformer.h.0.mlp.c_fc']
x = ctx.analysis[:1]
with torch.no_grad():
    h = cpu_model.transformer.wte(x) + cpu_model.transformer.wpe(torch.arange(x.shape[1]))
    xin = cpu_model.transformer.h[0].ln_2(h)
    y_sim, y_int = m(xin), integer_projection_reference(xin, m)
print(f'integer reference vs simulated: max |diff| = {float((y_sim - y_int).abs().max()):.2e} on outputs of magnitude up to {float(y_sim.abs().max()):.1f}')
del qm_cpu, cpu_model

In [ ]:
ptq = run_linear_ptq(ctx)
def results_frame(res, label):
    rows = []
    for c, r in res.items():
        p, st = r['profile'], r['storage']
        rows.append({'config': f'{label} {c}', 'alpha': r.get('alpha'), 'acc_%': r['lm']['accuracy'], 'nll': r['lm']['nll'], 'ppl': r['lm']['ppl'],
                     'int_export_MB': st['integer_export_mb'], 'packed_theory_MB': st['theoretical_packed_mb'], 'complete_runtime_MiB': st['simulated_runtime_fp32_mb'],
                     'latency_ms': p['latency_ms_mean'], 'latency_std': p['latency_ms_std'], 'tok/s': p['tokens_per_s'], 'gpu_MB': p.get('peak_gpu_allocated_mb'),
                     'rss_MB': p.get('peak_process_rss_mb_sampled'), 'cpu_%': p.get('avg_process_cpu_percent'), 'gpu_%': p.get('avg_gpu_util_percent'),
                     'gpu_W': p.get('avg_gpu_power_w'), 'mJ/pass': p.get('gpu_energy_per_pass_mj'), 'calib_s': r.get('calibration_seconds'),
                     'E_W': r['error_summary']['E_W'], 'E_X': r['error_summary']['E_X'], 'E_Y_w': r['error_summary']['E_Y_w'], 'E_Y': r['error_summary']['E_Y'], 'clip': r['error_summary']['clip_frac']})
    return pd.DataFrame(rows).set_index('config')
display(results_frame(ptq, 'Linear PTQ'))
display(pd.DataFrame({f'{c} {k}': {pt: v[k] for pt, v in r['error_summary']['per_type'].items()} for c, r in ptq.items() for k in ('E_X', 'E_Y')}).T)
display(Markdown(_static_dynamic_text(ptq)))

In [ ]:
# layer-wise errors for all 48 projections
lw = pd.DataFrame({c: {n.replace('transformer.h.', 'h'): e['E_Y'] for n, e in r['layer_errors'].items()} for c, r in ptq.items()})
display(lw.style.format('{:.2e}').background_gradient(axis=None))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 3.5))
lw.plot(ax=ax, logy=True, marker='.', lw=0.8)
ax.set_ylabel('E_Y (W + A8)')
ax.set_xticks(range(len(lw)))
ax.set_xticklabels(lw.index, rotation=90, fontsize=6)
plt.tight_layout()
plt.show()
for f in ptq['static_W4A8'].get('figures', []):
    display(Image(f, width=900))

In [ ]:

stats_orig = run_distribution_stats(ctx)
display(pd.DataFrame(stats_table(stats_orig)).set_index('projection'))
sq = run_smoothquant(ctx, stats_orig)
print('floating-point equivalence of the smoothed model (before quantization):')
display(pd.DataFrame(sq['equivalence']).T)
print('alpha selection on validation perplexity:')
display(pd.DataFrame({c: {a: v['ppl'] for a, v in s.items()} for c, s in sq['alpha_search'].items()}).T.assign(selected=pd.Series(sq['selected_alpha'])))
display(results_frame(sq['configs'], 'SmoothQuant'))
for k in sorted(k for k in sq if k.startswith('stats_smoothed_alpha_')):
    print('after smoothing,', k)
    display(pd.DataFrame(sq[k]).set_index('projection'))
display(Markdown(_smoothquant_text(ptq, sq)))

In [ ]:
# before vs after smoothing: per-channel maxima, histograms, and the quantized/dequantized approximation of the pre-quantization tensors
for f in sq['figures']:
    if 'dist_' in f or '_h5_' in f:
        display(Image(f, width=1000))
print('all distribution figures are in', cfg.figures)

In [ ]:
gq = run_gptq(ctx, stats_orig)
for k in ('W8', 'W4'):
    q = gq['quantization'][k]
    print(f"GPTQ {k}: {q['total_seconds']:.1f} s total, block size {q['block_size']}, percdamp {q['percdamp']}, ordering {q['ordering']}, "
          f"work device {q['work_device']}, peak GPU alloc {q.get('peak_gpu_allocated_mb')} MB, process RSS {q.get('peak_process_rss_mb'):.0f} MB, "
          f"Cholesky retries {sum(l['cholesky_retries'] for l in q['layers'])}, dead inputs {sum(l['dead_inputs'] for l in q['layers'])}")
display(pd.DataFrame(gq['quantization']['W4']['layers']).set_index('name').head(8))
display(results_frame(gq['configs'], 'GPTQ'))
cmp = pd.DataFrame({f'{m} {c}': {k: r['error_summary'][k] for k in ('E_W', 'E_X', 'E_Y_w', 'E_Y')} for m, res in (('RTN', ptq), ('SmoothQuant', sq['configs']), ('GPTQ', gq['configs'])) for c, r in res.items()}).T
print('reconstruction errors on identical validation inputs (E_Y_w = layer-output error from weights only, E_Y = with A8):')
display(cmp)
for f in gq['figures']:
    if '_h5_' in f: display(Image(f, width=800))
display(Markdown(_gptq_text(ptq, sq, gq)))

In [ ]:
rows = consolidated_rows(t1, ptq, sq, gq)
cons = pd.DataFrame(rows).set_index('configuration')
display(cons.drop(columns=['execution']))
cons.to_csv(cfg.root / 'consolidated_table.csv')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cons['ppl'].plot.barh(ax=axes[0], title='test perplexity')
axes[0].axvline(cons['ppl'].iloc[0], color='k', ls='--', lw=0.8)
cons['size_mb'].plot.barh(ax=axes[1], title='serialized size (MiB, integer export)')
cons['latency_ms'].plot.barh(ax=axes[2], title='latency (ms, batch 1, simulated)', xerr=cons['latency_std_ms'])
plt.tight_layout()
fig.savefig(cfg.figures / 'consolidated.png', dpi=110)
display(Image(filename=str(cfg.figures / 'consolidated.png'), width=1100))
plt.close(fig)
display(Markdown(_consolidated_text(rows, t1)))

In [ ]:
# Release GPT-2 before ResNet profiling. Only the deployed model remains on GPU.
del ctx, stats_orig
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
t3 = run_task3(qcfg)
rows = []
for tag, r in t3['models'].items():
    p = r['profile']
    st = r.get('storage', {})
    rows.append({'model': tag, 'train_top1': r['train']['top1'], 'test_top1': r['test']['top1'], 'test_top5': r['test']['top5'], 'test_macroF1': r['test']['macro_f1'], 'val_top1': r['val']['top1'],
                 'export_MB': st.get('integer_export_mb', r['serialized_fp32_mb']), 'packed_theory_MB': st.get('theoretical_packed_mb', r['serialized_fp32_mb']), 'fp32_state_MB': r['serialized_fp32_mb'],
                 'latency_ms': p['latency_ms_mean'], 'latency_std': p['latency_ms_std'], 'img/s': p['items_per_s'], 'gpu_MB': p.get('peak_gpu_allocated_mb'), 'rss_MB': p.get('peak_process_rss_mb_sampled'),
                 'cpu_%': p.get('avg_process_cpu_percent'), 'gpu_%': p.get('avg_gpu_util_percent'), 'gpu_W': p.get('avg_gpu_power_w'), 'mJ/pass': p.get('gpu_energy_per_pass_mj'),
                 'E_W': r.get('error_summary', {}).get('E_W'), 'E_X': r.get('error_summary', {}).get('E_X'), 'train_s': r.get('train_seconds'), 'execution': r['execution']})
display(pd.DataFrame(rows).set_index('model'))
print('MACs per 32x32 image:', t3['macs'])
for tag in ('qat_w8a8', 'qat_w4a8'):
    h = t3['histories'][tag]
    print(tag, 'best epoch', h['best_epoch'], 'config', h['config'])
    display(pd.DataFrame({k: h[k] for k in ('epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr', 'epoch_seconds', 'observer_frozen')}).set_index('epoch'))
display(Image(t3['figures'][0], width=900))
display(pd.DataFrame({tag: {n: e['E_W'] for n, e in t3['models'][tag]['layer_errors'].items()} for tag in ('qat_w8a8', 'qat_w4a8')}).rename(columns=lambda c: 'E_W ' + c)
        .join(pd.DataFrame({tag: {n: e['E_X'] for n, e in t3['models'][tag]['layer_errors'].items()} for tag in ('qat_w8a8', 'qat_w4a8')}).rename(columns=lambda c: 'E_X ' + c)))
display(Markdown(_qat_text(t3)))

In [ ]:
report_path = PROJECT / ('REPORT_smoke.md' if SMOKE else 'REPORT.md')
text = build_report(cfg.root, report_path)
print(f'wrote {report_path} ({len(text.splitlines())} lines)')
display(Markdown(text.split('# Task 1')[0]))
print('All task results and report generated from this execution.')
# Record the executed dependency versions and export the report/result artifacts.
import importlib.metadata, zipfile
packages = ['torch','torchvision','transformers','datasets','timm','huggingface_hub','numpy','pandas','matplotlib','scikit-learn','psutil','nvidia-ml-py','nbformat','nbconvert','ipykernel','tabulate']
versions = {p: importlib.metadata.version(p) for p in packages}
requirements_path = PROJECT / 'requirements.txt'
requirements_path.write_text('# Exact packages in the successful Colab run. Keep torch/torchvision CUDA builds compatible.\n' + '\n'.join(f'{p}=={v}' for p,v in versions.items()) + '\n')
save_json({'versions': versions, 'python': sys.version, 'execution_method': 'Google Colab Run all', 'smoke': SMOKE, 'reuse': cfg.reuse}, cfg.root / 'runtime_versions.json')
archive_path = PROJECT / ('PA3_smoke_outputs.zip' if SMOKE else 'PA3_experiment_outputs.zip')
export_paths = [report_path, requirements_path] + [p for p in cfg.root.rglob('*') if p.is_file() and p.suffix not in ('.pt','.pth','.bin','.pyc')]
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in export_paths:
        archive.write(path, path.resolve().relative_to(PROJECT))
print(f'Exported {archive_path.name}: {archive_path.stat().st_size:,} bytes')
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(archive_path))


In [ ]:
for label, group in [('Linear PTQ',ptq),('SmoothQuant',sq['configs']),('GPTQ',gq['configs'])]:
    print(label, 'layer-wise E_X')
    display(pd.DataFrame({c:{n.replace('transformer.h.','h'):e['E_X'] for n,e in r['layer_errors'].items()}
                          for c,r in group.items()}))
print('All size columns computed with 2**20 bytes use MiB.')
